# PointPillars (3D) → YOLOE-26 open-vocabulary (2D) → nhận xét từng vật

Notebook chạy liền một mạch trên `mini_val` của nuScenes (scene-0103, scene-0916, 81 keyframe):

```
LiDAR ──► PointPillars ──► box 3D ──► chấm điểm nuScenes (mAP, NDS) + xem 3D
                                 │
                                 ▼  với TỪNG box 3D
          chiếu lên 6 camera → chọn camera nhiều thông tin nhất (ít che, không cắt mép, rõ nét)
                                 │
                                 ▼
          YOLOE-26 (10 lớp nuScenes đặt bằng prompt) + số điểm lidar trong box
                                 │
                                 ▼
                         kết luận + nhận xét
```

| Phần | Cell |
|---|---|
| Cài đặt, dữ liệu, trọng số PointPillars | 1–6 |
| **A. PointPillars tạo box 3D**, chấm điểm, xem 3D | 7–9 |
| **B. YOLOE-26 kiểm chứng từng box**, kết luận, nhận xét | 10–15 |

**Vì sao YOLOE-26 (open-vocabulary)?** YOLO thường chỉ có 80 lớp COCO, thiếu `traffic_cone`, `barrier`, `trailer`, `construction_vehicle`, và định nghĩa lớp cũng khác nuScenes (ví dụ xe bán tải: COCO gọi là `car`, nuScenes gọi là `truck`). YOLOE cho phép **tự đặt lớp bằng chữ**, nên ở đây đặt **đủ 10 lớp của nuScenes**, mỗi lớp bằng một nhóm prompt bám theo định nghĩa của nuScenes.

**Các kết luận:**

| Kết luận | Nghĩa |
|---|---|
| **DUNG** | Camera thấy vật ở đúng vị trí, cùng lớp |
| **DUNG VAT, BOX LECH** | Đúng vật, đúng lớp, nhưng box 3D lệch rõ so với box 2D |
| **SAI LOP** | Đúng vị trí nhưng YOLOE gọi là lớp khác |
| **NGHI BAO NHAM** | Ảnh rõ mà YOLOE không thấy gì, và trong box 3D gần như không có điểm lidar |
| **CAMERA KHONG XAC NHAN** | Ảnh rõ mà YOLOE không thấy, nhưng lidar có điểm: có thể YOLOE bỏ sót, hoặc box 3D sai vị trí |
| **CHUA DU THONG TIN** | Ảnh tốt nhất vẫn bị che hoặc quá mờ |

**Nhãn gốc nuScenes không được dùng để kết luận.** Nó chỉ được in kèm (cột `tham_chieu_nhan_goc`) để bạn kiểm tra các kết luận có đáng tin không.

> ⚠️ **Bắt buộc dùng runtime có GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
>
> MMDetection3D được cài vào **môi trường Python 3.10 riêng** (Colab chạy Python 3.13, quá mới với nó). Phần đọc dữ liệu, YOLOE, kiểm chứng và vẽ chạy trong kernel Colab bình thường.

**Dữ liệu:** chỉ cần file `v1.0-mini.tgz` trong Google Drive. Chạy lần lượt Cell 1 → 15 (**Runtime → Run all** cũng được, nhưng Cell 3 sẽ hỏi quyền truy cập Drive).

## Cell 1 — Kiểm tra GPU

Nếu báo lỗi không có GPU: menu **Runtime → Change runtime type → T4 GPU → Save**, rồi chạy lại cell này.

In [ ]:
import importlib.util, subprocess, sys

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                    '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise RuntimeError(
        'Runtime nay KHONG co GPU. Vao menu Runtime -> Change runtime type -> T4 GPU -> Save, '
        'roi chay lai tu cell nay.')

print('GPU    :', r.stdout.strip())
print('Python :', sys.version.split()[0])
assert importlib.util.find_spec('google.colab') is not None, \
    'Notebook nay danh cho Google Colab (colab.research.google.com).'
print('Kernel : Colab OK')

## Cell 2 — Cài đặt

Colab giờ chạy **Python 3.13**, còn bộ MMDetection3D (mmcv 2.x, mmdet 3.x, mmdet3d 1.4) chỉ hỗ trợ tới Python 3.11. Cài thẳng vào kernel Colab sẽ hỏng ở nhiều chỗ: không có torchvision phù hợp, `openmim` kéo `setuptools` về bản 60 không chạy được trên Python 3.12+, và `mmcv` phải build từ source.

Cách làm ở đây:
- **Kernel Colab** cài `nuscenes-devkit` và `plotly` để đọc dữ liệu và vẽ, cùng `ultralytics` (YOLOE-26) và `CLIP` (mã hoá prompt chữ cho YOLOE) cho phần kiểm chứng 2D. `nuscenes-devkit` được cài **không kèm phụ thuộc**, vì nó khai báo cần numpy dưới bản 2 và sẽ hạ numpy của Colab, làm hỏng opencv. Thực tế phần devkit mà notebook dùng chạy tốt với numpy 2.
- **MMDetection3D** được cài vào môi trường riêng `/content/mmenv` chạy **Python 3.10**, tạo bằng `uv`. Ở đó dùng tổ hợp đã được kiểm chứng và **có sẵn wheel**: PyTorch 2.1.2 + CUDA 11.8, mmcv 2.1.0, mmdet 3.3.0, mmdet3d 1.4.0. Không phải build gì, không phải vá giới hạn phiên bản. `setuptools` được ghim dưới bản 70 vì mmengine và mim còn dùng `pkg_resources`.

Mất khoảng **5–8 phút**. Mỗi bước được kiểm tra ngay, lỗi ở bước nào sẽ dừng và in log của bước đó. Log đầy đủ ở `/content/install.log`.

Nếu một lần cài trước đó đã hạ numpy của kernel, cell này sẽ nâng lại lên numpy 2. Khi đó cell sẽ báo **cần restart**: vào menu **Runtime → Restart session**, rồi chạy lại từ Cell 1. Lần chạy lại cell này chỉ mất vài giây vì phần đã cài được bỏ qua.

In [ ]:
import os, subprocess, sys, time

ENV  = '/content/mmenv'           # moi truong Python 3.10 rieng cho MMDetection3D
PY   = f'{ENV}/bin/python'
LOG  = '/content/install.log'
DONE = f'{ENV}/.install_ok'


def run(desc, cmd):
    """Chay mot lenh cai dat; dung ngay va in log neu lenh that bai."""
    t0 = time.time()
    print(f'>> {desc} ...', end=' ', flush=True)
    with open(LOG, 'a') as f:
        f.write(f'\n\n$ {cmd}\n')
        f.flush()
        rc = subprocess.run(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print('LOI\n')
        print(open(LOG).read()[-4000:])
        raise RuntimeError(f'Buoc "{desc}" that bai. Log day du: {LOG}')
    print(f'xong ({time.time() - t0:.0f}s)')


# 1) Kernel Colab: devkit de doc du lieu, plotly de ve, uv de tao moi truong rieng.
#    --no-deps: devkit khai bao numpy<2 nhung thuc te chay tot voi numpy 2. Cai kem phu thuoc
#    se ha numpy cua Colab xuong 1.26 va lam hong opencv, jax... (opencv moi can numpy>=2).
#    Colab da co san cv2, matplotlib, sklearn, PIL, tqdm, cachetools; chi thieu pyquaternion.
run('Cai nuscenes-devkit (khong kem phu thuoc) + pyquaternion',
    f'{sys.executable} -m pip install -q --no-deps nuscenes-devkit pyquaternion')
# Giu numpy/shapely o ban ma Colab can (sua lai neu lan cai truoc da ha phien ban).
run('Cai plotly, uv; giu numpy 2 va shapely 2.1 cho kernel',
    f'{sys.executable} -m pip install -q plotly uv "numpy>=2.1,<2.6" "shapely>=2.1"')

# YOLOE-26 cho phan kiem chung 2D, chay trong kernel Colab. CLIP dung de ma hoa prompt chu.
run('Cai ultralytics (YOLOE-26) + CLIP',
    f'{sys.executable} -m pip install -q ultralytics git+https://github.com/ultralytics/CLIP.git')

# 2) Moi truong Python 3.10 rieng cho MMDetection3D.
if os.path.exists(DONE):
    print('Moi truong MMDetection3D da cai truoc do, bo qua.')
else:
    UV = f'{sys.executable} -m uv pip install --python {PY}'
    MMCV_WHEELS = 'https://download.openmmlab.com/mmcv/dist/cu118/torch2.1.0/index.html'

    run('Tao moi truong Python 3.10',
        f'{sys.executable} -m uv venv {ENV} --python 3.10 --seed --clear')
    run('Cai PyTorch 2.1.2 + CUDA 11.8',
        f'{UV} torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118')
    # --only-binary mmcv: bat buoc dung wheel dung san, khong bao gio build tu source.
    # setuptools<70: mmengine va mim can pkg_resources, ban setuptools moi da bo module nay.
    PINS = '"numpy<2" "setuptools<70"'
    run('Cai mmengine + mmcv 2.1.0',
        f'{UV} {PINS} "mmengine>=0.10.3,<0.11" mmcv==2.1.0 '
        f'--find-links {MMCV_WHEELS} --only-binary mmcv')
    run('Cai mmdet 3.3.0 + mmdet3d 1.4.0 + openmim',
        f'{UV} {PINS} mmdet==3.3.0 mmdet3d==1.4.0 openmim '
        f'--find-links {MMCV_WHEELS} --only-binary mmcv')
    open(DONE, 'w').close()

# 3) Neu pip vua doi phien ban numpy ma kernel da nap ban khac, phai restart kernel.
if 'numpy' in sys.modules:
    on_disk = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                             capture_output=True, text=True).stdout.strip()
    loaded = sys.modules['numpy'].__version__
    if on_disk != loaded:
        raise RuntimeError(f'CAN RESTART KERNEL: numpy da doi tu {loaded} sang {on_disk}. '
                         'Vao menu Runtime -> Restart session, roi chay lai tu Cell 1.')

print('\nCai dat xong. Chay tiep cell kiem tra ben duoi.')

In [ ]:
import subprocess, sys

ENV = '/content/mmenv'
PY  = f'{ENV}/bin/python'

# Kiem tra moi truong MMDetection3D (Python 3.10).
CHECK = """
import sys, torch, numpy, mmengine, mmcv, mmdet, mmdet3d
from mmcv.ops import Voxelization          # op CUDA ma PointPillars can
print('python  ', sys.version.split()[0])
print('torch   ', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('numpy   ', numpy.__version__)
print('mmengine', mmengine.__version__)
print('mmcv    ', mmcv.__version__)
print('mmdet   ', mmdet.__version__)
print('mmdet3d ', mmdet3d.__version__)
assert torch.cuda.is_available(), 'Khong thay GPU: chon server Colab co GPU T4 (xem Cell 1)'
"""
r = subprocess.run([PY, '-c', CHECK], capture_output=True, text=True)
print('--- Moi truong MMDetection3D ---')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise RuntimeError('Moi truong MMDetection3D chua dung. Xem loi o tren, '
                       'hoac xoa /content/mmenv roi chay lai Cell 2.')

# Kiem tra kernel Colab.
import numpy, plotly, nuscenes
print('--- Kernel Colab ---')
print('python  ', sys.version.split()[0])
print('numpy   ', numpy.__version__)
print('plotly  ', plotly.__version__)
print('nuscenes-devkit OK')
import ultralytics, clip
print('ultralytics', ultralytics.__version__, '| CLIP OK')

## Cell 3 — Dữ liệu nuScenes từ Google Drive

Cell này:
1. **Mount Google Drive**. Lần đầu Colab sẽ hỏi quyền truy cập, chọn tài khoản chứa file và bấm **Allow**.
2. Tìm `v1.0-mini.tgz` theo đường dẫn `TGZ_PATH`. Nếu không có ở đó, cell tự tìm trong toàn bộ **My Drive**.
3. **Giải nén chỉ phần notebook cần** vào `/content/nuscenes` (ổ đĩa của runtime, đọc nhanh hơn Drive nhiều):
   - `v1.0-mini/*.json`: bảng annotation và metadata
   - `samples/LIDAR_TOP`: 404 lần quét lidar của các keyframe
   - `sweeps/LIDAR_TOP`: các lần quét xen giữa keyframe, dùng để gộp 10 sweep
   - `maps/*.png`: devkit cần các file này khi khởi tạo
   - `samples/CAM_*`: ảnh keyframe của **6 camera** (khoảng 420 MB), cần cho phần kiểm chứng 2D

   Dữ liệu radar được bỏ qua. Vẫn phải đọc hết file 4 GB nên mất khoảng **3–6 phút**. Lần chạy lại trên cùng runtime sẽ bỏ qua bước này.

Nếu file của bạn nằm ở chỗ khác, sửa `TGZ_PATH` (ví dụ `/content/drive/MyDrive/datasets/v1.0-mini.tgz`).

Dữ liệu trong `/content` mất khi runtime bị thu hồi. File trên Drive vẫn còn, chỉ cần chạy lại cell này.

In [ ]:
import glob, os, subprocess, time, os.path as osp
from google.colab import drive
from nuscenes.nuscenes import NuScenes
from nuscenes.utils.splits import create_splits_scenes

TGZ_PATH = '/content/drive/MyDrive/v1.0-mini.tgz'  #@param {type:"string"}
DATAROOT = '/content/nuscenes'     # giai nen vao o dia cua runtime, doc nhanh hon Drive
N_LIDAR_EXPECTED = 404             # so file lidar keyframe cua v1.0-mini
N_SWEEPS = 10                      # so lan quet gop them truoc keyframe, giong luc huan luyen
DONE = osp.join(DATAROOT, '.extract_cam_ok')
CAMERAS = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT', 'CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT']

# Chi lay phan notebook can: bo radar va anh camera giua cac keyframe.
MEMBERS = (['v1.0-mini/*', 'maps/*', 'samples/LIDAR_TOP/*', 'sweeps/LIDAR_TOP/*']
           + [f'samples/{c}/*' for c in CAMERAS])

if osp.exists(DONE):
    print('Du lieu da giai nen truoc do, bo qua.')
else:
    if not osp.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    if not osp.isfile(TGZ_PATH):
        print(f'Khong thay {TGZ_PATH}, dang tim v1.0-mini.tgz trong My Drive ...')
        found = glob.glob('/content/drive/MyDrive/**/v1.0-mini.tgz', recursive=True)
        if not found:
            raise FileNotFoundError(
                'Khong tim thay v1.0-mini.tgz trong My Drive. Kiem tra file da upload xong chua, '
                'hoac sua TGZ_PATH cho dung duong dan roi chay lai cell nay.')
        TGZ_PATH = found[0]
    print(f'Giai nen {TGZ_PATH} ({osp.getsize(TGZ_PATH) / 1e9:.1f} GB) vao {DATAROOT} ...')
    t0 = time.time()
    os.makedirs(DATAROOT, exist_ok=True)
    r = subprocess.run(['tar', '-xzf', TGZ_PATH, '-C', DATAROOT, '--wildcards', *MEMBERS],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-4000:])
        raise RuntimeError('Giai nen that bai. File .tgz tren Drive co the bi hong hoac chua upload xong.')
    print(f'Giai nen xong ({time.time() - t0:.0f}s)')

n_lidar = len(glob.glob(osp.join(DATAROOT, 'samples', 'LIDAR_TOP', '*.pcd.bin')))
n_maps = len(glob.glob(osp.join(DATAROOT, 'maps', '*.png')))
print(f'File lidar keyframe: {n_lidar}/{N_LIDAR_EXPECTED} | file ban do: {n_maps}/4')
n_cams = {c: len(glob.glob(osp.join(DATAROOT, 'samples', c, '*.jpg'))) for c in CAMERAS}
print('Anh camera keyframe:', n_cams)
assert n_lidar == N_LIDAR_EXPECTED and n_maps == 4 and all(n == N_LIDAR_EXPECTED for n in n_cams.values()), \
    'Thieu file sau khi giai nen: xoa /content/nuscenes (!rm -rf /content/nuscenes) roi chay lai cell nay.'

nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=False)
print('Nap xong:', len(nusc.scene), 'scene,', len(nusc.sample), 'keyframe')

# Moi keyframe cua mini_val can N_SWEEPS lan quet lien truoc no.
needed = set()
for s in nusc.sample:
    if nusc.get('scene', s['scene_token'])['name'] not in create_splits_scenes()['mini_val']:
        continue
    sd = nusc.get('sample_data', s['data']['LIDAR_TOP'])
    for _ in range(N_SWEEPS):
        if not sd['prev']:
            break
        sd = nusc.get('sample_data', sd['prev'])
        needed.add(sd['filename'])
missing = sorted(f for f in needed if not osp.exists(osp.join(DATAROOT, f)))
print(f'Lan quet can cho mini_val: {len(needed) - len(missing)}/{len(needed)} co tren runtime')
for f in missing[:10]:
    print('  thieu:', f)
assert not missing, (f'Thieu {len(missing)} file sweep: xoa /content/nuscenes '
                     '(!rm -rf /content/nuscenes) roi chay lai cell nay.')
open(DONE, 'w').close()

## Cell 4 — Tải trọng số PointPillars đã huấn luyện

Dùng `mim download` để lấy đúng file cấu hình và trọng số khớp với phiên bản mmdet3d đang cài. Mô hình: **PointPillars + FPN**, huấn luyện trên tập train đầy đủ của nuScenes (700 scene), 10 lớp.

In [ ]:
import glob, subprocess

ENV = '/content/mmenv'
CKPT_DIR = '/content/checkpoints'
MODEL_NAME = 'pointpillars_hv_fpn_sbn-all_8xb4-2x_nus-3d'

if not glob.glob(f'{CKPT_DIR}/*.pth'):
    subprocess.run([f'{ENV}/bin/mim', 'download', 'mmdet3d',
                    '--config', MODEL_NAME, '--dest', CKPT_DIR], check=True)

cfg_file  = glob.glob(f'{CKPT_DIR}/*.py')[0]
ckpt_file = glob.glob(f'{CKPT_DIR}/*.pth')[0]
print('config    :', cfg_file)
print('checkpoint:', ckpt_file)

## Cell 5 — Chuẩn bị bộ suy luận

Model chạy trong môi trường Python 3.10, nên cell này ghi ra một script nhỏ `/content/pp_infer.py`. Notebook gọi script đó với danh sách file point cloud và nhận lại box qua file `.npz`. Mỗi lần gọi, model chỉ nạp một lần rồi suy luận lần lượt cho tất cả các file.

Pipeline gốc của mmdet3d gộp sweep bằng `LoadPointsFromMultiSweeps`, nhưng bước này cần file annotation `.pkl` riêng của mmdet3d. Vì vậy **notebook tự gộp sweep ở Cell 6, theo đúng cách của bước đó**, rồi bỏ bước đó khỏi pipeline. Mỗi điểm đưa vào model có 4 cột `[0, 1, 2, 4]` = `(x, y, z, Δt)`. Model nuScenes này **không dùng cường độ phản xạ** (cột 3).

Script trả về cho mỗi box: tâm, kích thước, góc xoay, **vận tốc** (model dự đoán được khi có nhiều lần quét) và 8 góc hộp.

Cell này nạp thử model một lần để kiểm tra, mất khoảng 10–20 giây.

In [ ]:
import subprocess, numpy as np

ENV = '/content/mmenv'
PY = f'{ENV}/bin/python'
WORKER = '/content/pp_infer.py'

WORKER_CODE = r"""
# Chay trong moi truong Python 3.10.  Cach goi:
#   python pp_infer.py CONFIG CHECKPOINT OUT.npz [frame1.bin frame2.bin ...]
# Moi file .bin la float32 (N, 5) = (x, y, z, intensity, dt), da gop sweep san.
import sys
import numpy as np
from mmdet3d.apis import init_model, inference_detector

cfg_file, ckpt_file, out_file = sys.argv[1:4]
bins = sys.argv[4:]

model = init_model(cfg_file, ckpt_file, device='cuda:0')

# Bo buoc gop sweep khoi pipeline (notebook da gop san), nhung giu cach no chon cot:
# [0, 1, 2, 4] = (x, y, z, dt). Model nuScenes nay KHONG dung intensity (cot 3).
full = model.cfg.test_dataloader.dataset.pipeline
sweeps = [t for t in full if t['type'] == 'LoadPointsFromMultiSweeps']
use_dim = list(sweeps[0].get('use_dim', [0, 1, 2, 4])) if sweeps else [0, 1, 2, 4]
sweeps_num = int(sweeps[0].get('sweeps_num', 10)) if sweeps else 0
n_in = int(model.cfg.model.pts_voxel_encoder.get('in_channels', 4))
assert len(use_dim) == n_in, f'use_dim {use_dim} khong khop in_channels={n_in} cua model'

pipeline = [t for t in full if t['type'] != 'LoadPointsFromMultiSweeps']
for t in pipeline:
    if t['type'] == 'LoadPointsFromFile':
        t['load_dim'], t['use_dim'] = 5, use_dim
model.cfg.test_dataloader.dataset.pipeline = pipeline

out = {'classes': np.array(model.dataset_meta['classes']),
       'pipeline': np.array([t['type'] for t in pipeline]),
       'use_dim': np.array(use_dim),
       'sweeps_num': np.array(sweeps_num)}
for i, path in enumerate(bins):
    result, _ = inference_detector(model, path)
    p = result.pred_instances_3d
    b = p.bboxes_3d                                                  # he toa do lidar
    vel = b.tensor[:, 7:9] if b.tensor.shape[1] >= 9 else b.tensor.new_zeros((len(b), 2))
    out[f'centers_{i}'] = b.gravity_center.cpu().numpy()             # (N, 3) tam hinh hoc
    out[f'dims_{i}'] = b.dims.cpu().numpy()                          # (N, 3) = (dai, rong, cao)
    out[f'yaws_{i}'] = b.yaw.cpu().numpy()                           # (N,) goc quanh truc z
    out[f'vels_{i}'] = vel.cpu().numpy()                             # (N, 2) van toc vx, vy
    out[f'corners_{i}'] = b.corners.cpu().numpy()                    # (N, 8, 3)
    out[f'labels_{i}'] = p.labels_3d.cpu().numpy()
    out[f'scores_{i}'] = p.scores_3d.cpu().numpy()
np.savez(out_file, **out)
"""
with open(WORKER, 'w') as f:
    f.write(WORKER_CODE)


def call_worker(out_file, bins=()):
    """Goi script suy luan trong moi truong Python 3.10, tra ve noi dung file .npz."""
    r = subprocess.run([PY, WORKER, cfg_file, ckpt_file, out_file, *bins],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:])
        print(r.stderr[-4000:])
        raise RuntimeError('Suy luan that bai, xem log o tren.')
    return np.load(out_file)


meta = call_worker('/content/_meta.npz')
CLASSES = [str(c) for c in meta['classes']]
print('10 lop model du doan:', CLASSES)
print('\nPipeline suy luan:', [str(t) for t in meta['pipeline']])
print('Cot dau vao moi diem:', meta['use_dim'].tolist(), '= (x, y, z, dt)')
print('So sweep luc huan luyen:', int(meta['sweeps_num']))
assert int(meta['sweeps_num']) == N_SWEEPS, \
    f"Config dung {int(meta['sweeps_num'])} sweep: sua N_SWEEPS o Cell 3 cho khop roi chay lai tu Cell 3"

## Cell 6 — Các hàm xử lý

- `load_points()` — **gộp lần quét của keyframe với N lần quét liền trước**, làm giống hệt `LoadPointsFromMultiSweeps` của mmdet3d:
  - Keyframe giữ nguyên, `Δt = 0`.
  - Với mỗi lần quét cũ hơn: bỏ các điểm trong ô vuông 1 m quanh cảm biến (điểm rơi trên nóc xe), chiếu về hệ toạ độ lidar của keyframe (vì xe đã di chuyển giữa hai lần quét), rồi đặt `Δt` = số giây mà lần quét đó cũ hơn keyframe.
  - Với `n_sweeps = 0`, hàm chỉ đọc lần quét của keyframe, như bản cũ của notebook.
- `run_pointpillars()` — suy luận nhiều keyframe trong một lần gọi, có lưu đệm.
- `check_box_convention()` — kiểm tra 8 góc hộp sau khi đổi quy ước **trùng khớp** với 8 góc do mmdet3d tính. Nếu quy ước kích thước hoặc góc xoay bị sai, mọi chỉ số chấm điểm sẽ vô nghĩa, nên notebook dừng ngay.
- `pred_to_global()` — đổi dự đoán từ hệ lidar sang **hệ toạ độ toàn cục** theo định dạng file kết quả của nuScenes, giống `NuScenesMetric` của mmdet3d. Hàm này cũng suy ra thuộc tính (đang đỗ, đang chạy...) từ vận tốc.
- `global_to_lidar_corners()` — chiều ngược lại, dùng để vẽ box cùng point cloud.
- `match_boxes()` — so khớp theo chuẩn nuScenes: cùng lớp, tâm cách nhau dưới 2 m trên mặt phẳng ngang, ưu tiên dự đoán có điểm tin cậy cao.

In [ ]:
import os
import numpy as np
from pyquaternion import Quaternion
from nuscenes.utils.data_classes import Box
from nuscenes.utils.geometry_utils import transform_matrix

FRAME_DIR = '/content/_frames'
os.makedirs(FRAME_DIR, exist_ok=True)
PRED_KEYS = ('centers', 'dims', 'yaws', 'vels', 'corners', 'labels', 'scores')
_PRED_CACHE = {}   # (sample token, so sweep) -> du doan tho cua model

# Thuoc tinh mac dinh theo lop, giong NuScenesMetric cua mmdet3d (model khong du doan thuoc tinh).
DEFAULT_ATTR = {'car': 'vehicle.parked', 'pedestrian': 'pedestrian.moving', 'trailer': 'vehicle.parked',
                'truck': 'vehicle.parked', 'bus': 'vehicle.moving', 'motorcycle': 'cycle.without_rider',
                'construction_vehicle': 'vehicle.parked', 'bicycle': 'cycle.without_rider',
                'barrier': '', 'traffic_cone': ''}


def get_sample(scene_name, index):
    scene = next(s for s in nusc.scene if s['name'] == scene_name)
    assert index < scene['nbr_samples'], \
        f"{scene_name} chi co {scene['nbr_samples']} keyframe"
    token = scene['first_sample_token']
    for _ in range(index):
        token = nusc.get('sample', token)['next']
    return nusc.get('sample', token), scene


def _lidar_poses(sample_data):
    """(calibrated_sensor, ego_pose) cua mot lan quet lidar."""
    return (nusc.get('calibrated_sensor', sample_data['calibrated_sensor_token']),
            nusc.get('ego_pose', sample_data['ego_pose_token']))


def _sensor_to_global(sample_data):
    cs, pose = _lidar_poses(sample_data)
    return (transform_matrix(pose['translation'], Quaternion(pose['rotation']))
            @ transform_matrix(cs['translation'], Quaternion(cs['rotation'])))


def load_points(sample, n_sweeps):
    """Point cloud (N, 5) = (x, y, z, intensity, dt) trong he toa do lidar cua keyframe.

    Lam giong LoadPointsFromMultiSweeps cua mmdet3d: keyframe giu nguyen voi dt = 0. Voi moi sweep
    truoc do: bo diem trong o vuong 1 m quanh cam bien, chieu ve he lidar cua keyframe
    (xe da di chuyen giua hai lan quet), dt = keyframe som hon sweep bao nhieu giay.
    """
    ref = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
    pts = np.fromfile(nusc.get_sample_data_path(ref['token']), dtype=np.float32).reshape(-1, 5).copy()
    pts[:, 4] = 0.0
    parts = [pts]
    global_to_ref = np.linalg.inv(_sensor_to_global(ref))
    sd = ref
    for _ in range(n_sweeps):
        if not sd['prev']:              # dau scene: co it sweep hon, model van chay duoc
            break
        sd = nusc.get('sample_data', sd['prev'])
        p = np.fromfile(nusc.get_sample_data_path(sd['token']), dtype=np.float32).reshape(-1, 5).copy()
        p = p[~((np.abs(p[:, 0]) < 1.0) & (np.abs(p[:, 1]) < 1.0))]
        tf = global_to_ref @ _sensor_to_global(sd)
        p[:, :3] = p[:, :3] @ tf[:3, :3].T + tf[:3, 3]
        p[:, 4] = (ref['timestamp'] - sd['timestamp']) * 1e-6
        parts.append(p)
    return np.concatenate(parts)


def run_pointpillars(samples, n_sweeps):
    """Du doan tho (chua loc) cho nhieu keyframe. Keyframe chua co trong bo dem duoc suy luan
    trong mot lan goi, nen model chi phai nap mot lan."""
    todo = [s for s in samples if (s['token'], n_sweeps) not in _PRED_CACHE]
    if todo:
        bins = []
        for s in todo:
            path = f"{FRAME_DIR}/{s['token']}_{n_sweeps}.bin"
            load_points(s, n_sweeps).tofile(path)
            bins.append(path)
        d = call_worker(f'{FRAME_DIR}/_batch.npz', bins)
        for i, s in enumerate(todo):
            _PRED_CACHE[(s['token'], n_sweeps)] = {k: d[f'{k}_{i}'] for k in PRED_KEYS}
        for path in bins:
            os.remove(path)
    return [_PRED_CACHE[(s['token'], n_sweeps)] for s in samples]


def _lidar_box(center, dims, yaw, vel=(0.0, 0.0)):
    """Box mmdet3d (dims = l, w, h; yaw quanh truc z) -> Box cua devkit (wlh), van trong he lidar."""
    return Box(center, dims[[1, 0, 2]], Quaternion(axis=[0, 0, 1], radians=float(yaw)),
               velocity=(float(vel[0]), float(vel[1]), 0.0))


def check_box_convention(pred, tol=0.01):
    """8 goc cua Box sau khi doi phai trung 8 goc do mmdet3d tinh. Neu khong, quy uoc kich thuoc
    hoac goc xoay bi sai va moi chi so cham diem se vo nghia."""
    for c, dim, yaw, k in zip(pred['centers'], pred['dims'], pred['yaws'], pred['corners']):
        ours = _lidar_box(c, dim, yaw).corners().T                       # (8, 3)
        gap = np.linalg.norm(ours[:, None, :] - k[None, :, :], axis=2).min(axis=1).max()
        assert gap < tol, f'Quy uoc box lech {gap:.3f} m so voi mmdet3d'
    return len(pred['centers'])


def _attribute(name, vel):
    """Giong NuScenesMetric cua mmdet3d: suy thuoc tinh tu toc do du doan."""
    if np.hypot(vel[0], vel[1]) > 0.2:
        if name in ('car', 'construction_vehicle', 'bus', 'truck', 'trailer'):
            return 'vehicle.moving'
        if name in ('bicycle', 'motorcycle'):
            return 'cycle.with_rider'
    else:
        if name == 'pedestrian':
            return 'pedestrian.standing'
        if name == 'bus':
            return 'vehicle.stopped'
    return DEFAULT_ATTR[name]


def pred_to_global(pred, sample):
    """Du doan (he lidar) -> danh sach box theo dinh dang file ket qua cua nuScenes (he toan cuc)."""
    cs, pose = _lidar_poses(nusc.get('sample_data', sample['data']['LIDAR_TOP']))
    out = []
    for c, dim, yaw, v, lab, sc in zip(pred['centers'], pred['dims'], pred['yaws'],
                                       pred['vels'], pred['labels'], pred['scores']):
        box = _lidar_box(c, dim, yaw, v)
        box.rotate(Quaternion(cs['rotation']))
        box.translate(np.array(cs['translation']))
        box.rotate(Quaternion(pose['rotation']))
        box.translate(np.array(pose['translation']))
        name = CLASSES[int(lab)]
        out.append({'sample_token': sample['token'],
                    'translation': box.center.tolist(),
                    'size': box.wlh.tolist(),
                    'rotation': box.orientation.elements.tolist(),
                    'velocity': box.velocity[:2].tolist(),
                    'detection_name': name,
                    'detection_score': float(sc),
                    'attribute_name': _attribute(name, v)})
    return out


def global_to_lidar_corners(eval_boxes, sample):
    """Box cua bo cham diem (he toan cuc) -> 8 goc trong he lidar cua keyframe, de ve cung point cloud."""
    cs, pose = _lidar_poses(nusc.get('sample_data', sample['data']['LIDAR_TOP']))
    corners = []
    for b in eval_boxes:
        box = Box(b.translation, b.size, Quaternion(b.rotation))
        box.translate(-np.array(pose['translation']))
        box.rotate(Quaternion(pose['rotation']).inverse)
        box.translate(-np.array(cs['translation']))
        box.rotate(Quaternion(cs['rotation']).inverse)
        corners.append(box.corners().T)
    return np.array(corners).reshape(-1, 8, 3)


def match_boxes(pred, gt, dist_th=2.0):
    """Khop tham lam theo diem tin cay giam dan, giong nuScenes: cung lop, tam cach nhau <= dist_th
    duoi dist_th tren mat phang ngang. pred, gt la danh sach DetectionBox."""
    pred_tp = np.zeros(len(pred), dtype=bool)
    gt_hit = np.zeros(len(gt), dtype=bool)
    pairs = []
    for pi in np.argsort([-b.detection_score for b in pred], kind='stable'):
        p = pred[pi]
        best, best_d = -1, dist_th
        for gi, g in enumerate(gt):
            if gt_hit[gi] or g.detection_name != p.detection_name:
                continue
            d = np.linalg.norm(np.array(p.translation[:2]) - np.array(g.translation[:2]))
            if d < best_d:
                best, best_d = gi, d
        if best >= 0:
            pred_tp[pi], gt_hit[best] = True, True
            pairs.append((pi, best, best_d))
    return pred_tp, gt_hit, pairs


print('Da dinh nghia xong cac ham.')

## Cell 7 — A. PointPillars tạo box 3D trên `mini_val`, chấm điểm chuẩn nuScenes

- **Suy luận** trên **81 keyframe**, dùng keyframe cộng 10 lần quét trước đó (**11 lần quét**, giống lúc model được huấn luyện).
- **Đổi box sang hệ toạ độ toàn cục** và ghi ra `results_nusc.json`.
- **Chấm điểm** bằng `DetectionEval` của devkit, là đúng bộ chấm mà nuScenes dùng cho bảng xếp hạng. Mất khoảng 2–4 phút.

Đặt `SAVE_TO_DRIVE = True` để chép file kết quả vào `MyDrive/pointpillars_out/`.

**Lưu ý:** `mini_val` **không có** trailer, construction_vehicle và barrier. Theo quy tắc nuScenes, lớp vắng mặt bị tính AP = 0, nên mAP cao nhất đạt được chỉ là 0.70. Vì vậy cell in thêm mAP tính riêng trên các lớp có mặt.

In [ ]:
import json, os, shutil, time
from nuscenes.eval.detection.config import config_factory
from nuscenes.eval.detection.evaluate import DetectionEval
from nuscenes.utils.splits import create_splits_scenes

SAVE_TO_DRIVE = False  #@param {type:"boolean"}
EVAL_SET = 'mini_val'
VAL_SCENES = create_splits_scenes()[EVAL_SET]
VAL_SAMPLES = [s for s in nusc.sample if nusc.get('scene', s['scene_token'])['name'] in VAL_SCENES]

t0 = time.time()
preds_raw = run_pointpillars(VAL_SAMPLES, N_SWEEPS)
n_checked = sum(check_box_convention(p) for p in preds_raw)
RESULTS = {s['token']: pred_to_global(p, s) for s, p in zip(VAL_SAMPLES, preds_raw)}

EVAL_DIR = f'/content/eval/{N_SWEEPS}sweep'
os.makedirs(EVAL_DIR, exist_ok=True)
RESULT_PATH = f'{EVAL_DIR}/results_nusc.json'
with open(RESULT_PATH, 'w') as f:
    json.dump({'meta': dict(use_camera=False, use_lidar=True, use_radar=False, use_map=False,
                            use_external=False), 'results': RESULTS}, f)
print(f'PointPillars: {len(VAL_SAMPLES)} keyframe, {n_checked} box (da kiem tra quy uoc box), '
      f'{time.time() - t0:.0f}s -> {RESULT_PATH}')

ev = DetectionEval(nusc, config=config_factory('detection_cvpr_2019'), result_path=RESULT_PATH,
                   eval_set=EVAL_SET, output_dir=EVAL_DIR, verbose=False)
metrics = ev.evaluate()[0].serialize()
present = [c for c in CLASSES if any(b.detection_name == c for b in ev.gt_boxes.all)]
print(f"\nmAP {metrics['mean_ap']:.3f} | NDS {metrics['nd_score']:.3f} | "
      f"mAP tren {len(present)} lop co mat: {np.mean([metrics['mean_dist_aps'][c] for c in present]):.3f}")
print('AP theo lop: ' + ', '.join(f"{c} {metrics['mean_dist_aps'][c]:.2f}" for c in present))

if SAVE_TO_DRIVE:
    if not osp.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/pointpillars_out', exist_ok=True)
    shutil.copy(RESULT_PATH, '/content/drive/MyDrive/pointpillars_out/results_nusc_10sweep.json')
    print('Da chep vao MyDrive/pointpillars_out/results_nusc_10sweep.json')

## Cell 8 — Nạp bộ kiểm chứng và lọc box 3D cần kiểm chứng

Cell đầu ghi ra `/content/verify_objects.py`, là **bản sao nguyên văn** của `verify_objects.py` trong thư mục dự án. Bản `.py` đó chạy được trên máy local bằng `python verify_objects.py --yoloe yoloe-26s-seg.pt`, nên chạy local hay Colab đều dùng chung một bộ code.

Cell sau giữ lại các box 3D thoả mãn cả ba điều kiện, giống bộ chấm nuScenes:
- điểm tin cậy ≥ `SCORE_THR`;
- nằm trong **phạm vi của lớp**: xe 50 m; người, xe đạp, xe máy 40 m; cọc, rào 30 m;
- không phải xe đạp nằm trong bãi đỗ xe đạp.

In [ ]:
%%writefile /content/verify_objects.py
"""Kiem chung tung box 3D (LiDAR) bang camera.

Voi moi box 3D:
  1. Chieu len ca 6 camera.
  2. Cham diem "luong thong tin" cua tung camera: phan nhin thay (khong bi cat mep), muc che khuat,
     do ro (kich thuoc, do net, do sang). Chon camera co diem cao nhat.
  3. Tren camera do: YOLO co thay vat khong, cung lop khong, box co khop khong; kem so diem lidar
     nam trong box 3D.
  4. Ket luan: DUNG / DUNG VAT, BOX LECH / SAI LOP / NGHI BAO NHAM / CAMERA KHONG XAC NHAN /
     CHUA DU THONG TIN.
Nhan goc nuScenes KHONG duoc dung de ket luan, chi in kem (cot "tham chieu") de doi chieu.

Chay:
    python verify_objects.py                                   # demo 1 keyframe, du doan mo phong
    python verify_objects.py --scene scene-0103 --keyframe 15
    python verify_objects.py --all                             # ca 81 keyframe cua mini_val
    python verify_objects.py --results path/to/results_nusc.json   # ket qua PointPillars that
    python verify_objects.py --yoloe yoloe-26s-seg.pt          # YOLOE open-vocabulary, du 10 lop nuScenes
Can: pip install ultralytics (kem torch); YOLOE can them CLIP (ultralytics tu cai lan dau).

Ket qua (out/verify/):
    <scene>_kf<k>_overview.png  - 6 camera, moi box 3D ve o camera duoc chon, mau theo ket luan
    <scene>_kf<k>_cards.png     - anh cat tung vat o camera duoc chon, kem ket luan
    ket_luan_<scene>_kf<k>.csv  - moi vat mot dong (--all: ket_luan_mini_val.csv)
"""
import argparse
import collections
import csv
import functools
import json
import os
import os.path as osp
import sys
import time

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.patches as patches  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402
from matplotlib.path import Path  # noqa: E402

# Doi duong dan bang bien moi truong khi dung o noi khac (vd. Colab: NUSC_DATAROOT=/content/nuscenes).
DATAROOT = os.environ.get('NUSC_DATAROOT', r'D:\AI_Action_Project')
VERSION = 'v1.0-mini'
OUT_DIR = os.environ.get('VERIFY_OUT_DIR', osp.join(DATAROOT, 'out', 'verify'))

if osp.isdir(osp.join(DATAROOT, 'nuscenes-devkit', 'python-sdk')):     # ban devkit trong thu muc du an
    sys.path.insert(0, osp.join(DATAROOT, 'nuscenes-devkit', 'python-sdk'))
from pyquaternion import Quaternion  # noqa: E402
from nuscenes.nuscenes import NuScenes  # noqa: E402
from nuscenes.eval.detection.config import config_factory  # noqa: E402
from nuscenes.eval.detection.utils import category_to_detection_name  # noqa: E402
from nuscenes.utils.data_classes import Box, LidarPointCloud  # noqa: E402
from nuscenes.utils.geometry_utils import points_in_box, transform_matrix, view_points  # noqa: E402
from nuscenes.utils.splits import create_splits_scenes  # noqa: E402

CAMERAS = ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
           'CAM_BACK_LEFT', 'CAM_BACK', 'CAM_BACK_RIGHT']       # thu tu nay = luoi 2 x 3 khi ve
CLASS_RANGE = config_factory('detection_cvpr_2019').class_range   # pham vi cham diem theo lop (m)

# --- Chieu box va chon camera ---
NEAR = 0.1              # mat phang gan truoc camera (m)
MIN_AREA = 100          # box 2D nho hon (px^2) thi khong xet camera do
EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
         (0, 4), (1, 5), (2, 6), (3, 7)]
SIZE_REF = 60.0         # canh box (px) tu muc nay tro len coi la du lon
SHARP_REF = 60.0        # phuong sai Laplacian tu muc nay tro len coi la du net
LIGHT_REF = 60.0        # do sang trung binh (0-255) tu muc nay tro len coi la du sang
CLEAR_OCC = 0.3         # "anh ro" = che < 30% ...
CLEAR_INFO = 0.5        # ... va diem thong tin >= 0.5; chi khi do moi dam ket luan camera KHONG thay vat

# --- Che khuat ---
MASK_SCALE = 1 / 8      # to mat na o 1/8 do phan giai cho nhanh
OCC_MARGIN = 1.0        # diem lidar gan camera hon mat truoc cua box qua 1 m moi tinh la vat che
MIN_LIDAR_PTS = 5       # it diem hon trong vung box thi khong dung cach lidar
DUP_DIST = 1.5          # hai box 3D cach nhau < 1.5 m coi la trung, khong tinh la che nhau

# --- Bo phat hien 2D ---
NUS_CLASSES = ['car', 'truck', 'bus', 'trailer', 'construction_vehicle', 'pedestrian',
               'motorcycle', 'bicycle', 'traffic_cone', 'barrier']           # 10 lop detection cua nuScenes

# YOLO thuong (80 lop COCO): chi 6 lop khop voi nuScenes.
COCO_TO_NUS = {'person': 'pedestrian', 'bicycle': 'bicycle', 'car': 'car',
               'motorcycle': 'motorcycle', 'bus': 'bus', 'truck': 'truck'}

# YOLOE (open-vocabulary): moi lop nuScenes la mot nhom prompt, chon theo dinh nghia lop cua nuScenes.
NUS_PROMPTS = {
    'car':                  ['car', 'sedan', 'suv', 'van', 'minivan'],             # nuScenes: car gom SUV, van
    'truck':                ['truck', 'pickup truck', 'box truck', 'lorry', 'semi truck'],   # ban tai la truck
    'bus':                  ['bus', 'city bus'],
    'trailer':              ['trailer', 'semi-trailer', 'cargo trailer'],
    'construction_vehicle': ['construction vehicle', 'excavator', 'bulldozer', 'crane', 'forklift'],
    'pedestrian':           ['pedestrian', 'person', 'child', 'construction worker', 'police officer'],
    'motorcycle':           ['motorcycle', 'motor scooter', 'moped', 'motorcyclist'],  # box bao ca nguoi lai
    'bicycle':              ['bicycle', 'cyclist'],                                     # box bao ca nguoi lai
    'traffic_cone':         ['traffic cone'],
    'barrier':              ['traffic barrier', 'concrete barrier', 'road barrier'],
}
# Prompt "doi thu": vat de bi nham voi cac lop tren (lan can co dinh khong phai barrier cua nuScenes),
# hoac vat nuScenes khong tinh (xe day, xe lan, scooter dung). Box cua chung bi bo, khong doi ve lop nao.
# Khong dung 'ambulance' / 'police car': thu thay chung hut mat xe tai / xe van that.
DISTRACTOR_PROMPTS = ['fence', 'guardrail', 'pole', 'traffic sign', 'fire hydrant', 'trash can', 'bollard',
                      'mailbox', 'stroller', 'wheelchair', 'kick scooter']

# Cau hinh bo phat hien dang dung (mac dinh: YOLO COCO). setup_open_vocab() doi sang YOLOE.
#   label_map: ten lop cua mo hinh -> lop nuScenes (None = bo)
#   checkable: lop nuScenes camera kiem duoc lop; matchable: lop nuScenes dem di so khop
#   class_conf: nguong diem tin cay rieng cho tung lop nuScenes
DETECTOR = {'kind': 'YOLO (COCO)', 'label_map': dict(COCO_TO_NUS),
            'checkable': set(COCO_TO_NUS.values()),
            'matchable': set(COCO_TO_NUS.values()) | {'trailer', 'construction_vehicle'},   # YOLO goi la truck
            'class_conf': {}}


def setup_open_vocab(model, prompts=None, distractors=None, class_conf=None):
    """Dat lop cho YOLOE bang prompt (du 10 lop nuScenes) va chuyen bo kiem chung sang che do nay."""
    prompts = NUS_PROMPTS if prompts is None else prompts
    distractors = DISTRACTOR_PROMPTS if distractors is None else distractors
    names = [p for ps in prompts.values() for p in ps] + list(distractors)
    assert len(set(names)) == len(names), 'Co prompt bi khai bao hai lan'
    model.set_classes(names)
    label_map = {p: c for c, ps in prompts.items() for p in ps}
    label_map.update({p: None for p in distractors})
    DETECTOR.update(kind='YOLOE (open-vocabulary)', label_map=label_map, checkable=set(prompts),
                    matchable=set(prompts), class_conf=dict(class_conf or {}))
    return names


RIDER_IOA = 0.3         # >= 30% dien tich nguoi nam trong box xe dap / xe may: coi la nguoi lai
IOU_THR = 0.4           # khop kieu IoU
IOA_THR = 0.7           # khop mot phan: >= 70% box YOLO nam trong box 3D ...
COVER_THR = 0.4         # ... va phu >= 40% phan nhin thay duoc cua box 3D
NEAR_Z = 8.0            # vat gan hon 8 m: box chieu phinh to hon vat that nhieu
DEPTH_MARGIN = 1.5      # sai so cho phep khi so do sau lidar (m)
CLASS_HEIGHT = {'car': 1.6, 'truck': 2.8, 'bus': 3.2, 'trailer': 3.5, 'construction_vehicle': 3.0,
                'pedestrian': 1.7, 'bicycle': 1.3, 'motorcycle': 1.3, 'traffic_cone': 0.7, 'barrier': 1.0}
RIDER_HEIGHT = 1.7
SHIFT_THR = 0.35        # tam lech ngang / day lech doc > 35% kich thuoc box YOLO: box lech
MIN_PTS_IN_BOX = 3      # so diem lidar toi thieu trong box 3D de coi la "co vat"
GT_DIST = 2.0           # doi chieu voi nhan goc: tam cach < 2 m (chuan nuScenes)

V_OK = 'DUNG'
V_SHIFT = 'DUNG VAT, BOX LECH'
V_CLASS = 'SAI LOP'
V_FP = 'NGHI BAO NHAM'
V_MISS = 'CAMERA KHONG XAC NHAN'
V_UNSURE = 'CHUA DU THONG TIN'
VERDICTS = [V_OK, V_SHIFT, V_CLASS, V_FP, V_MISS, V_UNSURE]
V_COLOR = {V_OK: 'limegreen', V_SHIFT: 'gold', V_CLASS: 'orange', V_FP: 'red',
           V_MISS: 'magenta', V_UNSURE: 'silver'}


# =============================================================================================
# Du lieu
# =============================================================================================
def ego_xy(nusc, sample):
    sd = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
    return np.array(nusc.get('ego_pose', sd['ego_pose_token'])['translation'][:2])


def bike_racks(nusc, sample):
    return [nusc.get_box(t) for t in sample['anns']
            if nusc.get('sample_annotation', t)['category_name'] == 'static_object.bicycle_rack']


def keep_box(name, center, xy, racks):
    """Giong bo cham diem nuScenes: trong pham vi cua lop, bo xe dap / xe may trong bai do xe dap."""
    if np.linalg.norm(np.asarray(center[:2]) - xy) > CLASS_RANGE[name]:
        return False
    return not (name in ('bicycle', 'motorcycle') and
                any(points_in_box(r, np.asarray(center, float).reshape(3, 1))[0] for r in racks))


def load_gt(nusc, sample):
    """Nhan goc thuoc 10 lop detection (chi de doi chieu, khong dung de ket luan)."""
    out = []
    for tok in sample['anns']:
        ann = nusc.get('sample_annotation', tok)
        name = category_to_detection_name(ann['category_name'])
        if name is None:
            continue
        b = nusc.get_box(tok)
        b.name, b.num_pts = name, ann['num_lidar_pts'] + ann['num_radar_pts']
        out.append(b)
    return out


SWAP = {'car': 'truck', 'truck': 'car', 'pedestrian': 'bicycle', 'bicycle': 'motorcycle',
        'motorcycle': 'bicycle', 'bus': 'truck', 'traffic_cone': 'barrier', 'barrier': 'traffic_cone',
        'trailer': 'truck', 'construction_vehicle': 'truck'}


def simulate_predictions(nusc, sample, rng):
    """Du doan mo phong tu nhan goc, co gai loi de kiem tra: lech vi tri (moi box), 10% bo sot,
    8% sai lop, 5% lech lon (1.5-3 m), them 2 box bao nham moi keyframe."""
    xy, out = ego_xy(nusc, sample), []
    for b in load_gt(nusc, sample):
        if b.num_pts == 0 or rng.random() < 0.10:
            continue
        p = b.copy()
        p.name = SWAP[b.name] if rng.random() < 0.08 else b.name
        ang = rng.uniform(0, 2 * np.pi)
        dist = rng.uniform(1.5, 3.0) if rng.random() < 0.05 else abs(rng.normal(0, 0.25))
        p.translate(np.array([np.cos(ang), np.sin(ang), 0.0]) * dist)
        p.score = float(rng.uniform(0.3, 0.9))
        out.append(p)
    z = nusc.get('ego_pose', nusc.get('sample_data', sample['data']['LIDAR_TOP'])['ego_pose_token'])['translation'][2]
    for _ in range(2):
        ang, r = rng.uniform(0, 2 * np.pi), rng.uniform(8, 35)
        out.append(Box([xy[0] + r * np.cos(ang), xy[1] + r * np.sin(ang), z + 0.8], [1.9, 4.5, 1.6],
                       Quaternion(axis=[0, 0, 1], angle=float(rng.uniform(0, 2 * np.pi))),
                       name='car', score=float(rng.uniform(0.3, 0.6))))
    return out


def load_predictions(nusc, samples, results_path, score_thr, seed):
    """{sample token: [Box he toan cuc]}, da loc theo diem tin cay, pham vi cua lop, bai do xe dap."""
    rng = np.random.default_rng(seed)
    if results_path:
        with open(results_path) as f:
            results = json.load(f)['results']
    out = {}
    for s in samples:
        if results_path:
            boxes = [Box(d['translation'], d['size'], Quaternion(d['rotation']),
                         name=d['detection_name'], score=d['detection_score'])
                     for d in results[s['token']] if d['detection_score'] >= score_thr]
        else:
            boxes = simulate_predictions(nusc, s, rng)
        xy, racks = ego_xy(nusc, s), bike_racks(nusc, s)
        out[s['token']] = [b for b in boxes if keep_box(b.name, b.center, xy, racks)]
    return out


def lidar_points(nusc, sample, nsweeps):
    """Diem lidar cua keyframe gop voi cac lan quet ngay truoc (da bu chuyen dong cua xe minh),
    trong he toan cuc (3, N). Mot lan quet rat thua o 20-30 m: nguoi di bo chi con 1-2 diem."""
    sd = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
    cs = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    pose = nusc.get('ego_pose', sd['ego_pose_token'])
    pc, _ = LidarPointCloud.from_file_multisweep(nusc, sample, 'LIDAR_TOP', 'LIDAR_TOP', nsweeps=nsweeps)
    pc.remove_close(1.0)
    tf = (transform_matrix(pose['translation'], Quaternion(pose['rotation']))
          @ transform_matrix(cs['translation'], Quaternion(cs['rotation'])))
    return tf[:3, :3] @ pc.points[:3] + tf[:3, 3:4]


def count_points_in_box(box, pts):
    """So diem lidar trong box 3D, bo lop mong sat day box (diem mat duong): 20 cm, hoac 15% chieu
    cao voi vat thap nhu coc giao thong."""
    local = box.orientation.inverse.rotation_matrix @ (pts - box.center.reshape(3, 1))
    w, l, h = box.wlh
    inside = ((np.abs(local[0]) <= l / 2) & (np.abs(local[1]) <= w / 2) &
              (local[2] <= h / 2) & (local[2] >= -h / 2 + min(0.2, 0.15 * h)))
    return int(inside.sum())


# =============================================================================================
# Chieu box, che khuat, do ro
# =============================================================================================
class Cam:
    """Thong so mot anh camera: cs, ego_pose luc chup, K, kich thuoc, ma tran global -> camera."""

    def __init__(self, nusc, cam_token):
        sd = nusc.get('sample_data', cam_token)
        self.cs = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
        self.pose = nusc.get('ego_pose', sd['ego_pose_token'])
        self.K = np.array(self.cs['camera_intrinsic'])
        self.w, self.h = sd['width'], sd['height']
        self.path = nusc.get_sample_data_path(cam_token)
        self.cam_from_global = (
            transform_matrix(self.cs['translation'], Quaternion(self.cs['rotation']), inverse=True)
            @ transform_matrix(self.pose['translation'], Quaternion(self.pose['rotation']), inverse=True))

    def box_to_cam(self, box):
        b = box.copy()
        b.translate(-np.array(self.pose['translation']))
        b.rotate(Quaternion(self.pose['rotation']).inverse)
        b.translate(-np.array(self.cs['translation']))
        b.rotate(Quaternion(self.cs['rotation']).inverse)
        return b

    def points_to_image(self, pts_global):
        """(u, v, do sau) cua cac diem nam trong anh."""
        p = self.cam_from_global[:3, :3] @ pts_global + self.cam_from_global[:3, 3:4]
        p = p[:, p[2] > NEAR]
        uv = view_points(p, self.K, normalize=True)[:2]
        ok = (uv[0] >= 0) & (uv[0] < self.w) & (uv[1] >= 0) & (uv[1] < self.h)
        return uv[0, ok], uv[1, ok], p[2, ok]


def project_box(box_cam, cam):
    """Box trong he camera -> thong tin 2D, hoac None neu khong lot vao anh. Cat box theo mat phang
    gan truoc khi chieu, vi goc nam sau camera chieu ra toa do sai."""
    k = box_cam.corners()
    front = k[2] > NEAR
    if not front.any():
        return None
    pts = [k[:, i] for i in range(8) if front[i]]
    for i, j in EDGES:
        if front[i] != front[j]:
            t = (NEAR - k[2, i]) / (k[2, j] - k[2, i])
            pts.append(k[:, i] + t * (k[:, j] - k[:, i]))
    pts = np.array(pts).T
    uv = view_points(pts, cam.K, normalize=True)[:2]
    x1, y1 = uv.min(axis=1)
    x2, y2 = uv.max(axis=1)
    cx1, cy1, cx2, cy2 = max(x1, 0.0), max(y1, 0.0), min(x2, cam.w), min(y2, cam.h)
    if cx2 <= cx1 or cy2 <= cy1:
        return None
    area = (cx2 - cx1) * (cy2 - cy1)
    return {'bbox': [float(cx1), float(cy1), float(cx2), float(cy2)],
            'vis': float(area / max((x2 - x1) * (y2 - y1), 1e-9)),   # phan nam trong anh
            'area': float(area), 'depth': float(box_cam.center[2]),
            'zmin': float(pts[2].min()), 'zmax': float(pts[2].max()),
            'hull': cv2.convexHull(uv.T.astype(np.float32)).reshape(-1, 2)}


def _mask(hull, shape):
    m = np.zeros(shape, np.uint8)
    cv2.fillConvexPoly(m, np.clip(hull * MASK_SCALE, -5000, 5000).round().astype(np.int32), 1)
    return m.astype(bool)


def _overlap(a, b):
    return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]


def add_occlusion(views, boxes, lidar_uvz, cam):
    """views: {chi so box: view} tren mot camera. Ghi 'occ' (muc che khuat, 0..1):
    - vung box co >= MIN_LIDAR_PTS diem lidar: ty le diem gan camera hon mat truoc cua box qua 1 m
      (do truc tiep vat chan tren duong nhin);
    - it diem hon: phan box bi box 3D khac phu len, voi box che phai gan camera hon it nhat 1 m
      (vat dung ngang hang canh nhau khong tinh la che).
    Chi dung du doan va lidar, khong dung nhan goc."""
    shape = (int(np.ceil(cam.h * MASK_SCALE)), int(np.ceil(cam.w * MASK_SCALE)))
    masks = {i: _mask(v['hull'], shape) for i, v in views.items()}
    u, vv, z = lidar_uvz
    for i, v in views.items():
        x1, y1, x2, y2 = v['bbox']
        sel = (u >= x1) & (u <= x2) & (vv >= y1) & (vv <= y2)
        inside = Path(v['hull']).contains_points(np.c_[u[sel], vv[sel]]) if sel.any() else []
        zin = z[sel][inside]
        if len(zin) >= MIN_LIDAR_PTS:
            v.update(occ=float((zin < v['zmin'] - OCC_MARGIN).mean()), occ_src='lidar')
            continue
        closer = np.zeros(shape, bool)
        for j, v2 in views.items():
            if (j != i and v2['depth'] < v['depth'] - OCC_MARGIN and _overlap(v['bbox'], v2['bbox'])
                    and np.linalg.norm(boxes[j].center - boxes[i].center) >= DUP_DIST):
                closer |= masks[j]
        n_own = masks[i].sum()
        v.update(occ=float((masks[i] & closer).sum() / n_own) if n_own else 0.0, occ_src='box')


def add_clarity(views, gray):
    """Do ro = kich thuoc x do net x do sang cua vung box tren anh xam."""
    for v in views.values():
        x1, y1, x2, y2 = (int(round(c)) for c in v['bbox'])
        crop = gray[y1:max(y2, y1 + 2), x1:max(x2, x1 + 2)]
        size = min(1.0, np.sqrt(v['area']) / SIZE_REF)
        sharp = float(cv2.Laplacian(crop, cv2.CV_64F).var()) if crop.size else 0.0
        light = float(crop.mean()) if crop.size else 0.0
        v.update(size=size, sharp=sharp, light=light,
                 clarity=size * max(0.3, min(1.0, sharp / SHARP_REF)) * max(0.3, min(1.0, light / LIGHT_REF)))
        v['info'] = v['vis'] * (1.0 - v['occ']) * v['clarity']


# =============================================================================================
# YOLO va so khop
# =============================================================================================
def _area(b):
    return max(b[2] - b[0], 0.0) * max(b[3] - b[1], 0.0)


def _inter(a, b):
    return _area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])


def _ioa(small, big):
    return _inter(small, big) / max(_area(small), 1e-9)


def iou_matrix(a, b):
    a, b = np.asarray(a, float).reshape(-1, 4), np.asarray(b, float).reshape(-1, 4)
    iw = np.clip(np.minimum(a[:, None, 2], b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(a[:, None, 3], b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    inter = iw * ih
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)


def merge_riders(dets):
    """nuScenes: box xe dap / xe may bao ca nguoi lai. Gop box 'person' dang ngoi tren xe vao box xe."""
    drop = set()
    for i, p in enumerate(dets):
        if p['name'] != 'pedestrian':
            continue
        cx = (p['bbox'][0] + p['bbox'][2]) / 2
        cands = [(_ioa(p['bbox'], c['bbox']), c) for c in dets
                 if c['name'] in ('bicycle', 'motorcycle') and not c['rider'] and c['bbox'][0] <= cx <= c['bbox'][2]]
        ioa, cyc = max(cands, key=lambda t: t[0], default=(0.0, None))
        if cyc is not None and ioa >= RIDER_IOA:
            a, b = cyc['bbox'], p['bbox']
            cyc.update(bbox=[min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])],
                       rider=True, conf=max(cyc['conf'], p['conf']))
            drop.add(i)
    return [d for i, d in enumerate(dets) if i not in drop]


def run_yolo(model, cams, imgsz, conf):
    """{cam: [box 2D]} cho 6 anh cua mot keyframe, lop da doi ve nuScenes theo DETECTOR.
    agnostic_nms: prompt gan nghia (car / suv) hoac prompt doi thu khong tao nhieu box cho cung vat."""
    names = model.names if isinstance(model.names, dict) else dict(enumerate(model.names))
    label_map, class_conf = DETECTOR['label_map'], DETECTOR['class_conf']
    keep = [i for i, n in names.items() if n in label_map]
    res = model.predict([cams[c].path for c in CAMERAS], imgsz=imgsz, conf=min([conf, *class_conf.values()]),
                        classes=keep, agnostic_nms=True, verbose=False)
    out = {}
    for c, r in zip(CAMERAS, res):
        dets = []
        for xyxy, cf, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()):
            label = names[int(k)]
            name = label_map[label]
            if name is not None and cf >= class_conf.get(name, conf):
                dets.append({'bbox': xyxy, 'conf': cf, 'name': name, 'label': label, 'rider': False})
        out[c] = merge_riders(dets)
    return out


def yolo_depth(d, lidar_uvz, cam):
    """(do sau, sai so cho phep) cua box YOLO: phan vi 30% do sau lidar trong box; khong du diem thi
    uoc luong tu chieu cao box (xe mau den thuong khong co diem lidar)."""
    u, v, z = lidar_uvz
    x1, y1, x2, y2 = d['bbox']
    sel = (u >= x1) & (u <= x2) & (v >= y1) & (v <= y2)
    if sel.sum() >= 3:
        return float(np.percentile(z[sel], 30)), DEPTH_MARGIN
    if y1 <= 1 or y2 >= cam.h - 1:
        return np.nan, np.nan
    zz = cam.K[1, 1] * (RIDER_HEIGHT if d['rider'] else CLASS_HEIGHT[d['name']]) / max(y2 - y1, 1.0)
    return zz, 0.3 * zz


def pair_scores(views, names, dets, depths):
    """Diem ghep (N, M) giua box 3D chieu xuong va box YOLO; > 0 la ghep duoc.
    IoU >= IOU_THR: diem = IoU. Khop mot phan (box 3D bi che / cat mep / o gan, box YOLO nam gon trong
    box 3D va phu du phan nhin thay, do sau khop): diem < 0.05. Cap khac lop phai co do sau khop."""
    iou = iou_matrix([v['bbox'] for v in views], [d['bbox'] for d in dets])
    score = np.where(iou >= IOU_THR, iou, 0.0)
    for i, v in enumerate(views):
        partial = v['occ'] >= 0.3 or v['vis'] < 0.95 or v['zmin'] < NEAR_Z
        visible_area = _area(v['bbox']) * max(1.0 - v['occ'], 0.2)
        for j, d in enumerate(dets):
            z, margin = depths[j]
            depth_ok = not np.isnan(z) and v['zmin'] - margin <= z <= v['zmax'] + margin
            if score[i, j] > 0:
                if names[i] != d['name'] and not np.isnan(z) and not depth_ok:
                    score[i, j] = 0.0
                continue
            if (partial and depth_ok and _ioa(d['bbox'], v['bbox']) >= IOA_THR
                    and _inter(d['bbox'], v['bbox']) / visible_area >= COVER_THR):
                score[i, j] = 0.01 + 0.01 * _ioa(d['bbox'], v['bbox'])
    return score, iou


def match_camera(views, boxes, dets, depths):
    """Hungarian tren mot camera: {chi so box: (chi so box YOLO, IoU, kieu khop)}."""
    from scipy.optimize import linear_sum_assignment
    idx = [i for i in views if boxes[i].name in DETECTOR['matchable']]
    if not idx or not dets:
        return {}
    score, iou = pair_scores([views[i] for i in idx], [boxes[i].name for i in idx], dets, depths)
    same = np.array([[boxes[i].name == d['name'] for d in dets] for i in idx])
    cost = np.where(score > 0, -(score + 0.05 * same), 1e6)
    out = {}
    for a, j in zip(*linear_sum_assignment(cost)):
        if score[a, j] > 0:
            out[idx[a]] = (int(j), float(iou[a, j]), 'IoU' if iou[a, j] >= IOU_THR else 'mot phan')
    return out


# =============================================================================================
# Ket luan
# =============================================================================================
def box_shift(v, d):
    """Do lech giua box 3D chieu xuong va box YOLO: (tam ngang, day) theo kich thuoc box YOLO."""
    (a1, b1, a2, b2), (c1, d1, c2, d2) = v['bbox'], d['bbox']
    w, h = max(c2 - c1, 1.0), max(d2 - d1, 1.0)
    return abs((a1 + a2) / 2 - (c1 + c2) / 2) / w, abs(b2 - d2) / h


def reference(boxes, gts):
    """Doi chieu voi nhan goc (chi de tham chieu): khop cung lop tam < 2 m (tham lam theo diem),
    neu khong thi nhan goc gan nhat bat ky lop."""
    ref, taken = [None] * len(boxes), set()
    for i in np.argsort([-b.score for b in boxes], kind='stable'):
        d = [(np.linalg.norm(boxes[i].center[:2] - g.center[:2]), k) for k, g in enumerate(gts)
             if k not in taken and g.name == boxes[i].name]
        if d and min(d)[0] < GT_DIST:
            taken.add(min(d)[1])
            ref[i] = 'dung (co vat that, cung lop)'
    for i, b in enumerate(boxes):
        if ref[i] is None:
            d = [(np.linalg.norm(b.center[:2] - g.center[:2]), g.name) for g in gts]
            ref[i] = (f'sai lop (nhan goc: {min(d)[1]})' if d and min(d)[0] < GT_DIST
                      else 'khong co vat that (bao nham)')
    return ref


def conclude(box, best_cam, v, match, d, mv, n_pts):
    """(ket luan, nhan xet) cho mot box 3D. v: view o camera duoc chon; match = (camera, (j, iou, kieu))
    neu YOLO khop; d, mv: box YOLO va view cua box 3D o camera khop."""
    pts_txt = f'{n_pts} diem lidar trong box'
    if best_cam is None:
        return V_UNSURE, f'Khong lot ro vao camera nao (chi lo o mep / qua nho). {pts_txt}.'
    view_txt = f'{best_cam}, nhin thay {v["vis"]:.0%}, che {v["occ"]:.0%}, do ro {v["clarity"]:.2f}'
    if match is not None:
        cam, (j, iou, kind) = match
        where = '' if cam == best_cam else f' (xac nhan o {cam})'
        if box.name not in DETECTOR['checkable']:
            return V_OK, (f'Camera thay vat{where} (YOLO goi la {d["name"]}); lop {box.name} YOLO khong co '
                          f'nen khong kiem duoc lop. {pts_txt}.')
        label = d.get('label', d['name'])                  # prompt / lop goc cua mo hinh 2D
        said = d['name'] if label.replace(' ', '_') == d['name'] else f"{d['name']} ('{label}')"
        if d['name'] != box.name:
            return V_CLASS, (f'Camera thay dung vi tri{where} nhung goi la {said} ({d["conf"]:.2f}), '
                             f'3D goi la {box.name}. {pts_txt}.')
        if kind == 'mot phan':
            return V_OK, f'Camera xac nhan{where}, vat bi che / cat mep nen chi khop mot phan. {pts_txt}.'
        dx, dy = box_shift(mv, d)
        if iou < 0.5 and (dx > SHIFT_THR or dy > SHIFT_THR):
            return V_SHIFT, (f'Camera thay dung vat, dung lop{where}, nhung box 3D lech '
                             f'(IoU {iou:.2f}, tam lech {dx:.0%}, day lech {dy:.0%} kich thuoc). {pts_txt}.')
        return V_OK, f'Camera xac nhan dung vat, dung lop{where} (IoU {iou:.2f}). {pts_txt}.'
    if box.name not in DETECTOR['checkable']:
        # Camera khong kiem duoc lop nay, nen khong du co so de ket luan bao nham.
        return V_UNSURE, f'YOLO khong co lop {box.name} nen camera khong kiem duoc. {pts_txt}.'
    if v['occ'] >= CLEAR_OCC or v['info'] < CLEAR_INFO:
        return V_UNSURE, f'Anh tot nhat van kem ({view_txt}); YOLO khong thay. {pts_txt}.'
    if n_pts < MIN_PTS_IN_BOX:
        return V_FP, f'Anh ro ({view_txt}) nhung YOLO khong thay gi, va {pts_txt}: nhieu kha nang bao nham.'
    return V_MISS, (f'Anh ro ({view_txt}) nhung YOLO khong thay; lidar van co {n_pts} diem. '
                    'Co the YOLO bo sot, hoac box 3D sai vi tri / kich thuoc.')


# =============================================================================================
# Xu ly mot keyframe
# =============================================================================================
def verify_sample(nusc, sample, boxes, model, args):
    cams = {c: Cam(nusc, sample['data'][c]) for c in CAMERAS}
    pts = lidar_points(nusc, sample, args.sweeps)
    lidar_uvz = {c: cams[c].points_to_image(pts) for c in CAMERAS}
    dets = run_yolo(model, cams, args.imgsz, args.conf)
    xy = ego_xy(nusc, sample)

    # 1) Chieu moi box len 6 camera, tinh che khuat va do ro.
    views = {c: {} for c in CAMERAS}
    for c, cam in cams.items():
        for i, b in enumerate(boxes):
            v = project_box(cam.box_to_cam(b), cam)
            if v is not None and v['area'] >= MIN_AREA:
                views[c][i] = v
        add_occlusion(views[c], boxes, lidar_uvz[c], cam)
        add_clarity(views[c], cv2.imread(cam.path, cv2.IMREAD_GRAYSCALE))

    # 2) So khop voi YOLO tren tung camera.
    matches = collections.defaultdict(dict)      # box -> {cam: (j, iou, kieu)}
    for c, cam in cams.items():
        depths = [yolo_depth(d, lidar_uvz[c], cam) for d in dets[c]]
        for i, m in match_camera(views[c], boxes, dets[c], depths).items():
            matches[i][c] = m

    # 3) Chon camera nhieu thong tin nhat, ket luan.
    refs = reference(boxes, [g for g in load_gt(nusc, sample)])
    rows = []
    for i, b in enumerate(boxes):
        ranked = sorted(((views[c][i]['info'], c) for c in CAMERAS if i in views[c]), reverse=True)
        best_cam = ranked[0][1] if ranked else None
        v = views[best_cam][i] if best_cam else None
        match = None
        if best_cam in matches[i]:
            match = (best_cam, matches[i][best_cam])
        elif matches[i]:
            c = max(matches[i], key=lambda c: views[c][i]['info'])
            match = (c, matches[i][c])
        n_pts = count_points_in_box(b, pts)
        d = dets[match[0]][match[1][0]] if match else None
        mv = views[match[0]][i] if match else None
        verdict, comment = conclude(b, best_cam, v, match, d, mv, n_pts)
        rows.append({
            'id': i, 'lop_3d': b.name, 'diem_3d': round(b.score, 2),
            'khoang_cach': round(float(np.linalg.norm(b.center[:2] - xy)), 1),
            'camera_chon': best_cam or '-',
            'cac_camera': ' > '.join(f'{c}:{s:.2f}' for s, c in ranked) or '-',
            'nhin_thay': round(v['vis'], 2) if v else None, 'che': round(v['occ'], 2) if v else None,
            'do_ro': round(v['clarity'], 2) if v else None, 'diem_thong_tin': round(v['info'], 2) if v else None,
            'lop_yolo': d['name'] if d else None, 'nhan_yolo': d.get('label') if d else None,
            'conf_yolo': round(d['conf'], 2) if d else None,
            'iou': round(match[1][1], 2) if match else None, 'kieu_khop': match[1][2] if match else None,
            'camera_khop': match[0] if match else None, 'diem_lidar': n_pts,
            'ket_luan': verdict, 'nhan_xet': comment, 'tham_chieu_nhan_goc': refs[i],
            'bbox_3d': [round(x) for x in v['bbox']] if v else None,
            'bbox_yolo': [round(x) for x in d['bbox']] if d else None,
        })
    return rows, cams, dets


# =============================================================================================
# Hinh ve
# =============================================================================================
def _rect(ax, b, color, ls='-', lw=2.0, dx=0, dy=0):
    ax.add_patch(patches.Rectangle((b[0] - dx, b[1] - dy), b[2] - b[0], b[3] - b[1], fill=False,
                                   edgecolor=color, linestyle=ls, linewidth=lw))


def draw_overview(rows, cams, title, out_png):
    fig, axes = plt.subplots(2, 3, figsize=(24, 10.5))
    for ax, c in zip(axes.ravel(), CAMERAS):
        ax.imshow(plt.imread(cams[c].path))
        sub = [r for r in rows if r['camera_chon'] == c]
        for r in sub:
            color = V_COLOR[r['ket_luan']]
            _rect(ax, r['bbox_3d'], color)
            if r['bbox_yolo'] and r['camera_khop'] == c:
                _rect(ax, r['bbox_yolo'], 'white', '--', 1.2)
            ax.text(r['bbox_3d'][0], r['bbox_3d'][1] - 4, f"#{r['id']} {r['lop_3d']}", fontsize=7.5,
                    bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))
        ax.set_title(f'{c}: {len(sub)} vat chon camera nay', fontsize=11)
        ax.set_xlim(0, cams[c].w)
        ax.set_ylim(cams[c].h, 0)
        ax.axis('off')
    handles = [Line2D([], [], color=V_COLOR[k], lw=3, label=k) for k in VERDICTS]
    handles.append(Line2D([], [], color='gray', lw=1.2, ls='--', label='box YOLO khop'))
    fig.legend(handles=handles, loc='lower center', ncol=len(handles), fontsize=11, frameon=False)
    fig.suptitle(title + '  |  moi vat chi ve o camera nhieu thong tin nhat, net lien = box 3D chieu xuong',
                 fontsize=13)
    fig.tight_layout(rect=(0, 0.03, 1, 0.97), h_pad=3.0)
    fig.savefig(out_png, dpi=80)
    plt.close(fig)


def draw_cards(rows, cams, title, out_png, ncol=6):
    rows = [r for r in rows if r['bbox_3d']]
    if not rows:
        return
    nrow = int(np.ceil(len(rows) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.9 * nrow), squeeze=False)
    imgs = {}
    for ax in axes.ravel():
        ax.axis('off')
    for ax, r in zip(axes.ravel(), rows):
        c = r['camera_chon']
        img = imgs.setdefault(c, plt.imread(cams[c].path))
        bs = [r['bbox_3d']] + ([r['bbox_yolo']] if r['bbox_yolo'] and r['camera_khop'] == c else [])
        x1, y1 = min(b[0] for b in bs), min(b[1] for b in bs)
        x2, y2 = max(b[2] for b in bs), max(b[3] for b in bs)
        pad = 0.3 * max(x2 - x1, y2 - y1) + 25
        cx1, cy1 = int(max(x1 - pad, 0)), int(max(y1 - pad, 0))
        cx2, cy2 = int(min(x2 + pad, cams[c].w)), int(min(y2 + pad, cams[c].h))
        ax.imshow(img[cy1:cy2, cx1:cx2])
        _rect(ax, r['bbox_3d'], V_COLOR[r['ket_luan']], dx=cx1, dy=cy1)
        if len(bs) == 2:
            _rect(ax, r['bbox_yolo'], 'white', '--', 1.3, dx=cx1, dy=cy1)
        label = r['nhan_yolo'] if r['nhan_yolo'] and r['nhan_yolo'].replace(' ', '_') != r['lop_yolo'] else ''
        yolo = (f"2D {r['lop_yolo']}{f' ({label})' if label else ''} {r['conf_yolo']:.2f}" if r['lop_yolo']
                else '2D: khong thay')
        ax.set_title(f"#{r['id']} 3D {r['lop_3d']} {r['diem_3d']:.2f} | {r['khoang_cach']:.0f} m\n"
                     f"{c[4:]} che {r['che']:.0%} ro {r['do_ro']:.2f} | lidar {r['diem_lidar']}\n"
                     f"{yolo}\n{r['ket_luan']}", fontsize=8.5, color='black',
                     bbox=dict(facecolor=V_COLOR[r['ket_luan']], alpha=0.35, pad=2, edgecolor='none'))
    fig.suptitle(title + '  |  net lien = box 3D chieu xuong, net dut trang = box YOLO khop', fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.98))
    fig.savefig(out_png, dpi=75)
    plt.close(fig)


# =============================================================================================
def get_sample(nusc, scene_name, index):
    scene = next(s for s in nusc.scene if s['name'] == scene_name)
    assert index < scene['nbr_samples'], f"{scene_name} chi co {scene['nbr_samples']} keyframe"
    token = scene['first_sample_token']
    for _ in range(index):
        token = nusc.get('sample', token)['next']
    return nusc.get('sample', token)


def main():
    ap = argparse.ArgumentParser(description=__doc__.split('\n')[0])
    ap.add_argument('--scene', default='scene-0916')
    ap.add_argument('--keyframe', type=int, default=20)
    ap.add_argument('--all', action='store_true', help='chay ca 81 keyframe cua mini_val (khong ve hinh)')
    ap.add_argument('--results', default=None, help='results_nusc.json cua PointPillars; bo trong = mo phong')
    ap.add_argument('--score-thr', type=float, default=0.3)
    ap.add_argument('--yolo', default='yolo26s.pt', help='YOLO thuong (80 lop COCO)')
    ap.add_argument('--yoloe', default=None,
                    help='dung YOLOE open-vocabulary thay cho --yolo, vd. yoloe-26s-seg.pt (du 10 lop nuScenes)')
    ap.add_argument('--imgsz', type=int, default=1280)
    ap.add_argument('--conf', type=float, default=0.25)
    ap.add_argument('--sweeps', type=int, default=5, help='so lan quet lidar gop lai (keyframe + truoc do)')
    ap.add_argument('--seed', type=int, default=0)
    args = ap.parse_args()

    from ultralytics import YOLO, YOLOE
    if args.yoloe:
        model = YOLOE(args.yoloe)
        setup_open_vocab(model)
    else:
        model = YOLO(args.yolo)
    nusc = NuScenes(version=VERSION, dataroot=DATAROOT, verbose=False)
    if args.all:
        val = create_splits_scenes()['mini_val']
        samples = [s for s in nusc.sample if nusc.get('scene', s['scene_token'])['name'] in val]
    else:
        samples = [get_sample(nusc, args.scene, args.keyframe)]
    preds = load_predictions(nusc, samples, args.results, args.score_thr, args.seed)
    source = args.results or 'MO PHONG (nhan goc + loi gai san)'
    print(f'Du doan 3D: {source} | {len(samples)} keyframe | {DETECTOR["kind"]}: {args.yoloe or args.yolo}, '
          f'imgsz {args.imgsz}')

    t0 = time.time()
    rows = verify_samples(nusc, samples, preds, model, args, draw=not args.all)
    if not args.all:
        print_objects(rows)
    print_summary(rows)
    save_csv(rows, osp.join(OUT_DIR, 'ket_luan_mini_val.csv' if args.all else
                            f'ket_luan_{args.scene}_kf{args.keyframe:02d}.csv'))
    print(f'({time.time() - t0:.0f}s)')


def verify_samples(nusc, samples, preds, model, args, draw=False, out_dir=None):
    """Kiem chung moi box 3D cua cac keyframe. preds: {sample token: [Box he toan cuc]};
    args can co imgsz, conf, sweeps. draw=True: luu hinh tong quan + anh cat cua tung keyframe."""
    out_dir = out_dir or OUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    all_rows, t0 = [], time.time()
    for n, s in enumerate(samples):
        scene_name = nusc.get('scene', s['scene_token'])['name']
        kf = _scene_tokens(nusc, s['scene_token']).index(s['token'])
        rows, cams, _ = verify_sample(nusc, s, preds[s['token']], model, args)
        for r in rows:
            r.update(scene=scene_name, keyframe=kf, sample_token=s['token'])
        all_rows += rows
        if draw:
            title = f'{scene_name} - keyframe {kf}'
            stem = osp.join(out_dir, f'{scene_name}_kf{kf:02d}')
            draw_overview(rows, cams, title, stem + '_overview.png')
            draw_cards(rows, cams, title, stem + '_cards.png')
            print(f'Da luu {stem}_overview.png va {stem}_cards.png')
        elif len(samples) > 1 and n % 10 == 0:
            print(f'  {n + 1}/{len(samples)} keyframe ({time.time() - t0:.0f}s)', flush=True)
    return all_rows


CSV_COLS = ['scene', 'keyframe', 'id', 'lop_3d', 'diem_3d', 'khoang_cach', 'camera_chon', 'cac_camera',
            'nhin_thay', 'che', 'do_ro', 'diem_thong_tin', 'lop_yolo', 'nhan_yolo', 'conf_yolo', 'iou', 'kieu_khop',
            'camera_khop', 'diem_lidar', 'ket_luan', 'nhan_xet', 'tham_chieu_nhan_goc']


def save_csv(rows, path):
    os.makedirs(osp.dirname(path), exist_ok=True)
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLS, extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)
    print(f'Da luu {path} ({len(rows)} vat)')


@functools.lru_cache(maxsize=None)
def _scene_tokens(nusc, scene_token):
    t, out = nusc.get('scene', scene_token)['first_sample_token'], []
    while t:
        out.append(t)
        t = nusc.get('sample', t)['next']
    return tuple(out)


def print_objects(rows):
    print(f"\n{'#':>3s} {'lop 3D':13s}{'diem':>5s}{'m':>5s}  {'camera chon':16s}{'che':>5s}{'ro':>5s}"
          f"{'thong tin':>10s}  {'YOLO':12s}{'IoU':>5s}{'lidar':>6s}  ket luan")
    print('-' * 118)
    for r in rows:
        yolo = f"{r['lop_yolo']} {r['conf_yolo']:.2f}" if r['lop_yolo'] else '-'
        iou = f"{r['iou']:.2f}" if r['iou'] is not None else '-'
        fmt = (lambda x: f'{x:5.2f}' if x is not None else '    -')
        print(f"{r['id']:3d} {r['lop_3d']:13s}{r['diem_3d']:5.2f}{r['khoang_cach']:5.0f}  {r['camera_chon']:16s}"
              f"{fmt(r['che'])}{fmt(r['do_ro'])}{fmt(r['diem_thong_tin']):>10s}  {yolo:12s}{iou:>5s}"
              f"{r['diem_lidar']:6d}  {r['ket_luan']}")
    print('\nNhan xet:')
    for r in rows:
        print(f"  #{r['id']:<3d} [{r['ket_luan']}] {r['nhan_xet']}  <tham chieu: {r['tham_chieu_nhan_goc']}>")


def print_summary(rows):
    n = len(rows)
    print(f'\n===== TONG KET: {n} box 3D =====')
    cnt = collections.Counter(r['ket_luan'] for r in rows)
    for k in VERDICTS:
        print(f'  {k:24s}{cnt[k]:6d}{cnt[k] / max(n, 1):8.0%}')
    print('\n===== DOI CHIEU KET LUAN VOI NHAN GOC (nhan goc KHONG duoc dung de ket luan) =====')
    refs = ['dung (co vat that, cung lop)', 'sai lop', 'khong co vat that (bao nham)']
    print(f"{'ket luan':24s}" + ''.join(f'{x[:22]:>24s}' for x in refs))
    for k in VERDICTS:
        sub = [r for r in rows if r['ket_luan'] == k]
        if sub:
            print(f'{k:24s}' + ''.join(
                f"{sum(r['tham_chieu_nhan_goc'].startswith(x[:7]) for r in sub):24d}" for x in refs))


if __name__ == '__main__':
    main()

In [ ]:
import importlib, sys

SCORE_THR = 0.3  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

os.environ['NUSC_DATAROOT'] = DATAROOT
os.environ['VERIFY_OUT_DIR'] = '/content/out/verify'
sys.path.insert(0, '/content')
import verify_objects as vo
importlib.reload(vo)

PREDS_3D = vo.load_predictions(nusc, VAL_SAMPLES, RESULT_PATH, SCORE_THR, seed=0)
print(f'{sum(map(len, PREDS_3D.values()))} box 3D can kiem chung tren {len(VAL_SAMPLES)} keyframe '
      f'(diem >= {SCORE_THR}, trong pham vi cua lop)')

## Cell 9 — Xem box 3D của PointPillars trên point cloud (3D tương tác)

**Điều khiển:**
- kéo chuột trái để xoay, lăn chuột để zoom, kéo chuột phải để tịnh tiến;
- bấm vào chú thích để ẩn/hiện, nháy đúp để đặt lại góc nhìn.

**Nội dung hình:**
- Box PointPillars (điểm ≥ `SCORE_THR`) được tô màu theo lớp. Di chuột lên box để xem số thứ tự, lớp và điểm tin cậy.
- **Nhãn gốc** vẽ bằng nét mảnh màu xám để so sánh.
- Point cloud gộp 11 lần quét, giống đầu vào của model.

Sau khi chạy Cell 12, Cell 14 vẽ lại hình này với box **tô màu theo kết luận**.

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
from pyquaternion import Quaternion

pio.renderers.default = 'colab'   # Colab web can renderer rieng cua plotly

SCENE_3D    = 'scene-0916'  #@param ['scene-0103', 'scene-0916']
KEYFRAME_3D = 20            #@param {type:"slider", min:0, max:39, step:1}
SHOW_GT_3D  = True          #@param {type:"boolean"}

EDGES = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4), (0, 4), (1, 5), (2, 6), (3, 7)]
CLASS_COLOR_3D = {'car': 'orange', 'truck': 'darkorange', 'bus': 'gold', 'trailer': 'khaki',
                  'construction_vehicle': 'yellow', 'pedestrian': 'deepskyblue', 'motorcycle': 'violet',
                  'bicycle': 'hotpink', 'traffic_cone': 'lime', 'barrier': 'white'}


def boxes_to_lidar(sample, boxes):
    """Box he toan cuc -> he toa do lidar cua keyframe (cung he voi load_points)."""
    sd = nusc.get('sample_data', sample['data']['LIDAR_TOP'])
    cs = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    pose = nusc.get('ego_pose', sd['ego_pose_token'])
    out = []
    for b in boxes:
        b = b.copy()
        b.translate(-np.array(pose['translation']))
        b.rotate(Quaternion(pose['rotation']).inverse)
        b.translate(-np.array(cs['translation']))
        b.rotate(Quaternion(cs['rotation']).inverse)
        out.append(b)
    return out


def wire_trace(boxes, color, name, texts=None, width=4):
    """Nhieu box -> mot trace khung day (12 canh + doan chi huong dau xe)."""
    xs, ys, zs, hover = [], [], [], []
    for n, b in enumerate(boxes):
        k = b.corners().T
        front = k[[0, 1, 2, 3]].mean(axis=0)
        for p, q in [(k[i], k[j]) for i, j in EDGES] + [(b.center, front)]:
            xs += [p[0], q[0], None]
            ys += [p[1], q[1], None]
            zs += [p[2], q[2], None]
            hover += [texts[n] if texts else name] * 3
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', line=dict(color=color, width=width),
                        name=f'{name} ({len(boxes)})', hovertext=hover, hoverinfo='text')


def show_boxes_3d(scene_name, index, rows=None, show_gt=True, max_points=80000, height=750):
    """Point cloud + box 3D can kiem chung. rows (ket qua kiem chung) co thi to mau theo ket luan,
    khong thi to mau theo lop."""
    sample = vo.get_sample(nusc, scene_name, index)
    pts = load_points(sample, N_SWEEPS)
    if len(pts) > max_points:
        pts = pts[np.random.default_rng(0).choice(len(pts), max_points, replace=False)]
    preds = boxes_to_lidar(sample, PREDS_3D[sample['token']])

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
                               marker=dict(size=1.1, color=pts[:, 2], colorscale='Viridis', opacity=0.55,
                                           cmin=-2.5, cmax=3),
                               name=f'lidar ({N_SWEEPS + 1} lan quet)', hoverinfo='skip'))
    if show_gt:
        fig.add_trace(wire_trace(boxes_to_lidar(sample, vo.load_gt(nusc, sample)), 'lightgray', 'nhan goc', width=2))
    if rows is None:
        for name in vo.NUS_CLASSES:
            idx = [i for i, b in enumerate(preds) if b.name == name]
            if idx:
                fig.add_trace(wire_trace([preds[i] for i in idx], CLASS_COLOR_3D[name], name,
                                         [f'#{i} {name} {preds[i].score:.2f}' for i in idx]))
    else:
        by_id = {r['id']: r for r in rows}
        for v in vo.VERDICTS:
            idx = [i for i in range(len(preds)) if by_id[i]['ket_luan'] == v]
            if idx:
                fig.add_trace(wire_trace([preds[i] for i in idx], vo.V_COLOR[v], v,
                                         [f"#{i} {preds[i].name} {preds[i].score:.2f}: {by_id[i]['nhan_xet']}"
                                          for i in idx]))
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers', marker=dict(size=6, color='red', symbol='x'),
                               name='xe (ego)'))
    what = 'to mau theo ket luan' if rows is not None else 'to mau theo lop'
    fig.update_layout(template='plotly_dark', height=height,
                      title=f'{scene_name} - keyframe {index} | {len(preds)} box PointPillars (diem >= {SCORE_THR}), {what}',
                      margin=dict(l=0, r=0, t=40, b=0), legend=dict(itemsizing='constant', font=dict(size=10)),
                      scene=dict(aspectmode='data', xaxis_title='x (m)', yaxis_title='y (m)', zaxis_title='z (m)',
                                 camera=dict(eye=dict(x=-1.4, y=-1.4, z=0.9))))
    fig.show()
    return fig


fig3d = show_boxes_3d(SCENE_3D, KEYFRAME_3D, show_gt=SHOW_GT_3D)

## Cell 10 — B. Cài YOLOE-26, đặt đủ 10 lớp nuScenes bằng prompt

**Mỗi lớp nuScenes là một nhóm prompt.** YOLOE trả về box theo prompt nào thì box đó được đổi về lớp nuScenes của prompt đó. Các prompt phụ bám theo **định nghĩa lớp của nuScenes**, để giảm lệch lớp giữa hai mô hình:

| Lớp nuScenes | Prompt | Lý do thêm prompt phụ |
|---|---|---|
| `car` | car, sedan, suv, van, minivan | nuScenes: car gồm cả SUV, van, minivan |
| `truck` | truck, pickup truck, box truck, lorry, semi truck | nuScenes: **xe bán tải là truck** |
| `bus` | bus, city bus | |
| `trailer` | trailer, semi-trailer, cargo trailer | |
| `construction_vehicle` | construction vehicle, excavator, bulldozer, crane, forklift | |
| `pedestrian` | pedestrian, person, child, construction worker, police officer | nuScenes gộp các loại người này vào pedestrian |
| `motorcycle` | motorcycle, motor scooter, moped, motorcyclist | xe tay ga là motorcycle; box **bao cả người lái** |
| `bicycle` | bicycle, cyclist | box **bao cả người lái** |
| `traffic_cone` | traffic cone | |
| `barrier` | traffic barrier, concrete barrier, road barrier | |

**Prompt "đối thủ"** (fence, guardrail, pole, traffic sign, fire hydrant, trash can, bollard, mailbox, stroller, wheelchair, kick scooter):
- Với open-vocabulary, mỗi box được gán cho **prompt giống nhất trong danh sách**. Nếu không có đối thủ, cột, hàng rào hay lan can dễ bị ép thành `traffic_cone` hay `barrier`. Có đối thủ thì chúng thắng, và box đó **bị bỏ đi**.
- nuScenes chỉ tính **rào chắn tạm thời** là `barrier`, không tính lan can cố định. Xe đẩy, xe lăn và scooter đứng cũng không nằm trong 10 lớp.
- **Không** dùng `ambulance` / `police car` làm đối thủ: khi thử, chúng "hút" mất cả xe tải và xe van thật.

**`CLASS_CONF`** là ngưỡng điểm tin cậy riêng cho từng lớp, vì điểm của mô hình open-vocabulary không đồng đều giữa các lớp. Chỉnh ngưỡng dựa vào Cell 11 và bảng đối chiếu ở Cell 12.

Có thể sửa `PROMPTS` / `DISTRACTORS` / `CLASS_CONF` ngay trong cell này. Lần chạy đầu sẽ tải trọng số YOLOE-26 và bộ mã hoá chữ, mất khoảng 1 phút.

In [ ]:
from ultralytics import YOLOE

YOLOE_MODEL = 'yoloe-26l-seg.pt'  #@param ['yoloe-26s-seg.pt', 'yoloe-26m-seg.pt', 'yoloe-26l-seg.pt', 'yoloe-26x-seg.pt']

PROMPTS = {
    'car':                  ['car', 'sedan', 'suv', 'van', 'minivan'],
    'truck':                ['truck', 'pickup truck', 'box truck', 'lorry', 'semi truck'],
    'bus':                  ['bus', 'city bus'],
    'trailer':              ['trailer', 'semi-trailer', 'cargo trailer'],
    'construction_vehicle': ['construction vehicle', 'excavator', 'bulldozer', 'crane', 'forklift'],
    'pedestrian':           ['pedestrian', 'person', 'child', 'construction worker', 'police officer'],
    'motorcycle':           ['motorcycle', 'motor scooter', 'moped', 'motorcyclist'],
    'bicycle':              ['bicycle', 'cyclist'],
    'traffic_cone':         ['traffic cone'],
    'barrier':              ['traffic barrier', 'concrete barrier', 'road barrier'],
}
DISTRACTORS = ['fence', 'guardrail', 'pole', 'traffic sign', 'fire hydrant', 'trash can', 'bollard',
               'mailbox', 'stroller', 'wheelchair', 'kick scooter']
CLASS_CONF = {c: 0.2 for c in PROMPTS}      # nguong rieng cho tung lop nuScenes
assert set(PROMPTS) == set(vo.NUS_CLASSES), 'PROMPTS phai co du 10 lop nuScenes'

t0 = time.time()
yoloe = YOLOE(YOLOE_MODEL)
names = vo.setup_open_vocab(yoloe, PROMPTS, DISTRACTORS, CLASS_CONF)
print(f'{YOLOE_MODEL}: {len(names)} prompt ({time.time() - t0:.0f}s)\n')
print(f"{'lop nuScenes':22s}{'nguong':>7s}   prompt")
print('-' * 95)
for c, ps in PROMPTS.items():
    print(f'{c:22s}{CLASS_CONF[c]:7.2f}   ' + ', '.join(ps))
print(f"{'(doi thu, bi bo)':22s}{'':7s}   " + ', '.join(DISTRACTORS))

## Cell 11 — Xem YOLOE-26 trên một keyframe (để chỉnh prompt / ngưỡng)

Cell chạy YOLOE trên 6 ảnh của keyframe đang chọn, **chưa so với box 3D**.

- Box có màu là box được giữ lại, nhãn dạng `lớp (prompt thắng) điểm`.
- Box **xám nét đứt** là box của **prompt đối thủ**, bị bỏ đi.

Nên xem vài keyframe khác nhau:
- Xe bán tải có ra `truck (pickup truck)` không?
- Cọc giao thông, rào chắn có được bắt không?
- Prompt đối thủ có "hút" nhầm vật thật không?

Nếu cần, sửa prompt hoặc ngưỡng ở Cell 10 rồi chạy lại Cell 10 và 11.

In [ ]:
import collections
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import Image, display

SCENE_2D    = 'scene-0916'  #@param ['scene-0103', 'scene-0916']
KEYFRAME_2D = 20            #@param {type:"slider", min:0, max:39, step:1}
IMGSZ       = 1280          #@param [640, 960, 1280, 1600] {type:"raw"}


def show_yoloe(scene_name, index, out_png='/content/out/yoloe_view.png'):
    sample = vo.get_sample(nusc, scene_name, index)
    paths = [nusc.get_sample_data_path(sample['data'][c]) for c in CAMERAS]
    res = yoloe.predict(paths, imgsz=IMGSZ, conf=min(CLASS_CONF.values()), agnostic_nms=True, verbose=False)
    fig, axes = plt.subplots(2, 3, figsize=(24, 10.5))
    counts = collections.Counter()
    for ax, cam, p, r in zip(axes.ravel(), CAMERAS, paths, res):
        ax.imshow(plt.imread(p))
        for xyxy, cf, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist()):
            prompt = r.names[int(k)]
            name = vo.DETECTOR['label_map'][prompt]
            if name is not None and cf < CLASS_CONF[name]:
                continue
            color = CLASS_COLOR_3D[name] if name else 'lightgray'
            ax.add_patch(patches.Rectangle((xyxy[0], xyxy[1]), xyxy[2] - xyxy[0], xyxy[3] - xyxy[1], fill=False,
                                           edgecolor=color, linewidth=2 if name else 1.2, linestyle='-' if name else '--'))
            label = (f"{name}{'' if prompt.replace(' ', '_') == name else f' ({prompt})'}" if name else prompt)
            ax.text(xyxy[0], xyxy[1] - 4, f'{label} {cf:.2f}', fontsize=7,
                    bbox=dict(facecolor=color, alpha=0.75, pad=1, edgecolor='none'))
            counts[name or f'(doi thu) {prompt}'] += 1
        ax.set_title(cam, fontsize=11)
        ax.axis('off')
    fig.suptitle(f'{scene_name} - keyframe {index} - {YOLOE_MODEL} (xam net dut = prompt doi thu, bi bo)', fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.97), h_pad=3.0)
    os.makedirs(osp.dirname(out_png), exist_ok=True)
    fig.savefig(out_png, dpi=80)
    plt.close(fig)
    display(Image(out_png))
    print('So box:', dict(counts.most_common()))


show_yoloe(SCENE_2D, KEYFRAME_2D)

## Cell 12 — B. Kiểm chứng mọi box 3D của PointPillars bằng YOLOE-26

Với **từng box 3D**:
1. **Chiếu lên 6 camera.** Box được cắt theo mặt phẳng trước camera, để các góc nằm sau camera không bị chiếu sai.
2. **Chấm điểm thông tin** của từng camera = phần nằm trong ảnh × (1 − mức che) × độ rõ, rồi **chọn camera điểm cao nhất**.
   - **Mức che:** trong vùng box, tỷ lệ điểm lidar (gộp `LIDAR_SWEEPS` lần quét) gần camera hơn mặt trước của box quá 1 m. Nếu vùng box có ít điểm lidar, dùng phần bị box 3D khác (gần hơn ≥ 1 m) phủ lên.
   - **Độ rõ:** kích thước box trên ảnh × độ nét (phương sai Laplacian) × độ sáng.
3. **Ghép với box YOLOE** bằng Hungarian: khớp khi IoU ≥ 0.4. Nếu vật bị che hoặc cắt mép, chấp nhận khớp một phần, miễn là độ sâu lidar khớp. Người ngồi trên xe đạp / xe máy được gộp vào box xe, theo quy ước của nuScenes.
4. **Kết luận** theo bảng ở đầu notebook. Kết luận "không thấy" (NGHI BAO NHAM, CAMERA KHONG XAC NHAN) **chỉ được đưa ra khi ảnh rõ**: bị che < 30% và điểm thông tin ≥ 0.5.

Cuối cell in bảng tổng kết, kèm bảng **đối chiếu với nhãn gốc** để xem mỗi loại kết luận đáng tin tới đâu. Mất khoảng 2–4 phút trên GPU T4.

In [ ]:
from types import SimpleNamespace

LIDAR_SWEEPS = 5  #@param {type:"slider", min:1, max:10, step:1}

ARGS = SimpleNamespace(imgsz=IMGSZ, conf=min(CLASS_CONF.values()), sweeps=LIDAR_SWEEPS)
print(f'Box 3D: PointPillars | 2D: {vo.DETECTOR["kind"]} {YOLOE_MODEL}, imgsz {IMGSZ}')
t0 = time.time()
ROWS = vo.verify_samples(nusc, VAL_SAMPLES, PREDS_3D, yoloe, ARGS)
print(f'Kiem chung xong {len(ROWS)} box ({time.time() - t0:.0f}s)')
vo.print_summary(ROWS)

## Cell 13 — Kết luận theo lớp, khoảng cách, camera được chọn; các cặp lệch lớp

In [ ]:
def crosstab(rows, key, keys):
    head = f"{'':22s}" + ''.join(f'{v[:14]:>16s}' for v in vo.VERDICTS) + f"{'TONG':>7s}"
    print(head)
    print('-' * len(head))
    for k in keys:
        sub = [r for r in rows if key(r) == k]
        if sub:
            print(f'{str(k):22s}' + ''.join(f"{sum(r['ket_luan'] == v for r in sub):16d}" for v in vo.VERDICTS)
                  + f'{len(sub):7d}')


BINS = [(0, 10), (10, 20), (20, 30), (30, 40), (40, 60)]
print('===== THEO LOP 3D =====')
crosstab(ROWS, lambda r: r['lop_3d'], vo.NUS_CLASSES)
print('\n===== THEO KHOANG CACH TOI XE =====')
crosstab(ROWS, lambda r: next(f'{a}-{b} m' for a, b in BINS if a <= r['khoang_cach'] < b),
         [f'{a}-{b} m' for a, b in BINS])
print('\n===== THEO CAMERA DUOC CHON =====')
crosstab(ROWS, lambda r: r['camera_chon'], CAMERAS + ['-'])

print('\n===== CAC CAP LECH LOP (3D -> YOLOE, prompt thang) =====')
pairs = collections.Counter((r['lop_3d'], r['lop_yolo'], r['nhan_yolo']) for r in ROWS if r['ket_luan'] == vo.V_CLASS)
for (a, b, p), n in pairs.most_common():
    print(f'  3D {a:22s} -> YOLOE {b:22s} (prompt "{p}")  x{n}')

## Cell 14 — Xem một keyframe: 3D tô màu theo kết luận, ảnh camera, nhận xét từng vật

1. **Hình 3D tương tác**: như Cell 9, nhưng box **tô màu theo kết luận**. Di chuột lên box để đọc nhận xét.
2. **Hình tổng quan 6 camera**: mỗi box 3D chỉ được vẽ ở **camera nhiều thông tin nhất** của nó. Nét đứt trắng là box YOLOE đã khớp.
3. **Ảnh cắt từng vật** ở camera được chọn, kèm mức che, độ rõ, số điểm lidar, lớp YOLOE (và prompt thắng) và kết luận.
4. **Bảng và nhận xét** cho từng vật, kèm tham chiếu nhãn gốc.

| Màu | Kết luận |
|---|---|
| 🟩 Xanh lá | DUNG |
| 🟨 Vàng | DUNG VAT, BOX LECH |
| 🟧 Cam | SAI LOP |
| 🟥 Đỏ | NGHI BAO NHAM |
| 🟪 Tím hồng | CAMERA KHONG XAC NHAN |
| ⬜ Xám | CHUA DU THONG TIN |

In [ ]:
SCENE    = 'scene-0916'  #@param ['scene-0103', 'scene-0916']
KEYFRAME = 20            #@param {type:"slider", min:0, max:39, step:1}

sample = vo.get_sample(nusc, SCENE, KEYFRAME)
rows_kf = [r for r in ROWS if r['sample_token'] == sample['token']]
vo.verify_samples(nusc, [sample], PREDS_3D, yoloe, ARGS, draw=True)   # ve hinh (ket qua giong ROWS)
show_boxes_3d(SCENE, KEYFRAME, rows=rows_kf, show_gt=False)
stem = f'/content/out/verify/{SCENE}_kf{KEYFRAME:02d}'
display(Image(stem + '_overview.png'))
display(Image(stem + '_cards.png'))
vo.print_objects(rows_kf)

## Cell 15 — Xuất kết quả ra CSV và tải về

Mỗi dòng là một box 3D của PointPillars, gồm:
- lớp, điểm tin cậy, khoảng cách;
- camera được chọn, kèm điểm thông tin của mọi camera (cột `cac_camera`);
- mức che, độ rõ;
- lớp YOLOE và **prompt thắng** (`nhan_yolo`), IoU, kiểu khớp;
- số điểm lidar;
- **kết luận**, **nhận xét**, tham chiếu nhãn gốc.

File mở được bằng Excel (UTF-8). Đặt `SAVE_CSV_TO_DRIVE = True` để chép thêm vào `MyDrive/pointpillars_out/`.

In [ ]:
from google.colab import files

SAVE_CSV_TO_DRIVE = False  #@param {type:"boolean"}
csv_path = f'/content/out/verify/ket_luan_pointpillars_yoloe_thr{SCORE_THR:.2f}.csv'
vo.save_csv(ROWS, csv_path)
if SAVE_CSV_TO_DRIVE:
    if not osp.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/pointpillars_out', exist_ok=True)
    shutil.copy(csv_path, '/content/drive/MyDrive/pointpillars_out/')
    print('Da chep vao MyDrive/pointpillars_out/')
files.download(csv_path)

---
## Cách đọc kết quả

- **Bảng đối chiếu với nhãn gốc** (cuối Cell 12) cho biết mỗi loại kết luận đáng tin tới đâu.
  - Khi thử với box mô phỏng có lỗi gài sẵn, **DUNG** và **SAI LOP** rất đáng tin (> 90%).
  - **NGHI BAO NHAM** kém tin cậy hơn: khoảng 40% khi dùng YOLO thường, vì bộ phát hiện 2D hay bỏ sót người đi bộ ở xa dù ảnh rõ. Hãy coi kết luận này là "cần xem lại", chưa phải bằng chứng chắc chắn.
- **SAI LOP**: xem bảng *các cặp lệch lớp* ở Cell 13.
  - Nếu một cặp lặp lại nhiều lần qua cùng một prompt (ví dụ 3D `car` → YOLOE `truck` qua prompt `van`), có thể là do prompt chứ không phải lỗi của PointPillars. Hãy sửa `PROMPTS` ở Cell 10.
  - Cột `tham_chieu_nhan_goc` cho biết bên nào đúng.
- **Open-vocabulary giúp kiểm tra được cả `traffic_cone`, `barrier`, `trailer`, `construction_vehicle`**. Với YOLO thường, các lớp này luôn rơi vào CHUA DU THONG TIN. Đổi lại, trên `car` và `pedestrian`, YOLOE có thể bỏ sót nhiều hơn YOLO thường một chút.
- `mini_val` chỉ có 2 scene, nên các tỷ lệ ở đây dùng để **tìm loại lỗi và minh hoạ**, chưa đủ để kết luận thống kê.

## Xử lý sự cố

| Lỗi | Cách xử lý |
|---|---|
| Không có GPU | Runtime → Change runtime type → T4 GPU → Save, rồi chạy lại từ Cell 1 |
| Một bước ở Cell 2 báo `LOI` | Đọc log in bên dưới (đầy đủ ở `/content/install.log`), khắc phục rồi chạy lại Cell 2 |
| Cell 2 báo numpy đã đổi phiên bản | Runtime → Restart session, rồi chạy lại từ Cell 1 |
| Cell 3 thiếu file sau khi giải nén | `!rm -rf /content/nuscenes` rồi chạy lại Cell 3 |
| `Quy uoc box lech ... m so voi mmdet3d` | Dùng đúng mmdet3d 1.4.0 như Cell 2 |
| `CUDA out of memory` | Runtime → Restart session, rồi chạy lại từ Cell 3 |
| Cell 10 báo lỗi tải `yoloe-26*.pt` hoặc bộ mã hoá chữ | Lỗi mạng: chạy lại Cell 10 |
| Cell 10 báo thiếu `clip` | Chạy `!pip install git+https://github.com/ultralytics/CLIP.git`, rồi chạy lại Cell 10 |
| `NameError` (`nusc`, `RESULT_PATH`, `vo`, `yoloe`…) | Runtime đã khởi động lại: chạy lại lần lượt từ Cell 3 |